# Eight years on Instagram — your own data, chapter by chapter

Notebooks 01 and 02 turned the raw export into an anonymised copy: `anonymised/csv/` holds one tidy table per kind of
event (messages, likes, follows, ads shown…) and `anonymised/json/` the pseudonymised export tree, with every person
replaced by a pseudonym. This notebook rebuilds, from *your* copy, the course's demo dashboard *Eight years on
Instagram*: the same thirteen chapters, the same charts with the same colours, and the same finding sentences — filled
with your numbers instead of the demo account's. Where the demo's sentence says "posting stopped", yours may say the
opposite: the sentences are templates, the data decide.

Everything runs inside your browser and nothing is uploaded. The data are pseudonymised: do not try to work out who
`user_0417` is — the point is the aggregate pattern, not the person.

Run the cells top to bottom with **Shift+Enter**. Not every export contains every table (a quiet account has no
`comments.csv`, an account that never watched Reels has an empty `videos_watched.csv`): every chapter checks what is
there and prints one line saying what it skips, rather than failing. The chart style lives in one cell (section 0b):
change a colour there and every chart follows.

## 0a. Load the data

The tables are in `anonymised/csv/` next to this notebook (notebook 02 wrote them there), the JSON tree in
`anonymised/json/`. If you only have the `anonymised.zip` you downloaded — on another computer, say — drop it into the
file browser on the left and this cell unzips it. Helpers: `load()` reads a table and returns `None` if it is not in
your export; `have()` checks that a table exists and has the columns a step needs; `to_dt()` parses timestamps
(Instagram writes them in UTC) and converts them to Rome time, so that "midnight" means midnight for you.

In [ ]:
import os, glob, zipfile, json, re, math, textwrap
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
import matplotlib.pyplot as plt

# 0a. Find anonymised/csv (or unzip anonymised.zip), looking in this folder first, then in the whole drive.
def csv_dir_in(folder):
    """The csv/ folder with tables inside an anonymised/ folder (also one level deeper, in case the zip had a top folder)."""
    for cand in [os.path.join(folder, "csv")] + glob.glob(os.path.join(folder, "*", "csv")):
        if os.path.isdir(cand) and glob.glob(os.path.join(cand, "*.csv")):
            return cand
    return None

def find_data():
    bases = [os.getcwd()] + ([r for r, _d, _f in os.walk("/drive")] if os.path.isdir("/drive") else [])
    for base in bases:
        folder, zip_path = os.path.join(base, "anonymised"), os.path.join(base, "anonymised.zip")
        found = csv_dir_in(folder) if os.path.isdir(folder) else None
        if found is None and os.path.exists(zip_path):
            print("Unzipping", zip_path)
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(folder)
            found = csv_dir_in(folder)
        if found:
            return found
    return None

DATA = find_data()
JSON_DIR = os.path.join(os.path.dirname(DATA), "json") if DATA else None   # the anonymised export tree, next to csv/
if JSON_DIR and not os.path.isdir(JSON_DIR):
    JSON_DIR = None
if DATA is None:
    print("No data found: run notebook 02 first, or drag your anonymised.zip into the file browser and re-run this cell.")
else:
    print("Data folder:", DATA, "| JSON tree:", JSON_DIR or "not found (chapters 2 and 10 lose some detail)")

TZ = "Europe/Rome"

def load(name):
    """Read anonymised/csv/<name>.csv as a DataFrame; None if the table is not in this export or is empty."""
    path = os.path.join(DATA or "", f"{name}.csv")
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
    except Exception as e:                       # empty or malformed file
        print(f"  (could not read {name}.csv: {e})")
        return None
    return df if len(df) else None

def have(df, *cols):
    """True if df exists, has rows, and has all the columns listed."""
    return df is not None and len(df) > 0 and all(c in df.columns for c in cols)

def to_utc(s):
    """Parse ISO timestamps (mixed with/without fractions) as UTC."""
    return pd.to_datetime(s, format="ISO8601", errors="coerce", utc=True)

def to_dt(s):
    """Same, converted to Rome time."""
    t = to_utc(s)
    try:
        return t.dt.tz_convert(TZ)
    except Exception:
        return t

def flag(s):
    """Robust boolean from a column that may be read as bool or as the strings 'True'/'False'."""
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

# 0b. List the tables with their row counts.
if DATA:
    rows = []
    for p in sorted(glob.glob(os.path.join(DATA, "*.csv"))):
        name = os.path.basename(p)[:-4]
        df = load(name)
        rows.append((name, 0 if df is None else len(df), "" if df is None else ", ".join(df.columns[:6]) + (" ..." if len(df.columns) > 6 else "")))
    display(pd.DataFrame(rows, columns=["table", "rows", "first columns"]).set_index("table"))

## 0b. The look of the charts — one cell to restyle everything

The dashboard uses eight series colours (`C[0]`…`C[7]`) and a muted grey, thin grey axes, a faint horizontal grid,
a dark card title with a grey note under it, and a small legend with square swatches below the chart. Every chart in
this notebook is drawn through the helpers defined here — `card()` opens a figure in that style, `legend_below()` adds
the legend, `pct_axis()` formats an axis as percentages — so changing a value here changes every chart. `fmtN` and
`fmtP` print numbers the way the page does (thousands separators; percentages rounded to whole points), and `say()`
prints a chapter's finding sentence with the placeholders `[calc.field]` filled from the results computed in `R`.

In [ ]:
import matplotlib as mpl
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator, NullLocator

# --- the dashboard's palette (dash.colors) and theme tokens, light theme
C = ["#2a78d6", "#d95926", "#199e70", "#c98500", "#d55181", "#008300", "#6250d6", "#e34948"]
MUTED = "#c3c2b7"          # --cds-chart-muted
AXIS = "#c3c2b7"           # axis and baseline lines
REF = "#97958d"            # dashed reference lines
TICK = "#898781"           # tick labels and notes
INK = "#0b0b0b"            # emphasised text
GRID = (0, 0, 0, 0.10)     # horizontal grid: black at 10 %
BORDER = (0, 0, 0, 0.10)   # card border
PAGE = "#f9f9f7"           # page background (cards are white)
WD = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
KIND_LABEL = {"post": "Posts published", "comment": "Comments written", "like": "Posts liked", "story_like": "Story reactions",
              "save": "Posts saved", "message": "Messages sent and received", "story": "Stories"}

mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9.5, "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
                     "xtick.color": TICK, "ytick.color": TICK, "xtick.labelsize": 9, "ytick.labelsize": 9, "axes.labelcolor": TICK,
                     "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
                     "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "figure.dpi": 100,
                     "xtick.major.size": 0, "ytick.major.size": 0, "xtick.minor.size": 0, "ytick.minor.size": 0,
                     "axes.grid": False, "legend.fontsize": 8.3})

def isna(v):
    try:
        return v is None or bool(pd.isna(v))
    except (TypeError, ValueError):
        return False

def fmtN(v):
    """d3.format(',') : thousands separators, integers without decimals."""
    if isna(v):
        return "—"
    v = v.item() if hasattr(v, "item") else v
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return f"{v:,}" if isinstance(v, (int, float)) else str(v)

def fmtP(v):
    """d3.format('.0%') : share -> whole percent, rounding half up like JavaScript's toFixed."""
    if isna(v):
        return "—"
    return f"{int(Decimal(float(v) * 100).quantize(Decimal(1), rounding=ROUND_HALF_UP))}%"

def fmt_field(field, v):
    """The page's fillText rule: *_pct / share* / top3 / top10 / top5 fields <= 1 are percentages, numbers get separators."""
    if isna(v):
        return "—"
    v = v.item() if hasattr(v, "item") else v
    if re.search(r"year|^y[01]$|_from$|^story_like_first$", str(field)) and isinstance(v, (int, float)) and not isinstance(v, bool):
        return str(int(v))                      # years as 2020, never 2,020
    if re.search(r"_pct$|^share|top3|top10|top5|_top5$", field) and isinstance(v, (int, float)) and not isinstance(v, bool) and v <= 1:
        return fmtP(v)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return fmtN(v)
    return str(v)

R = {}   # every calc's result, by the dashboard's calc id: a dict (summary) or a DataFrame (rows)

def say(template):
    """Print a finding sentence: [calc.field] and [calc.field@row] placeholders filled from R (row -1 = last row)."""
    def sub(m):
        src, field, row = m.group(1), m.group(2), m.group(3)
        data = R.get(src)
        if data is None:
            return "—"
        try:
            if row is not None:
                v = data.iloc[int(row)][field]
            else:
                v = data.get(field)
        except Exception:
            return "—"
        return fmt_field(field, v)
    print(textwrap.fill(re.sub(r"\[(\w+)\.(\w+)(?:@(-?\d+))?\]", sub, template), 112))

def skip(what, why):
    print(f"{what}: skipped — {why}.")

def run(fn, needs=(), where="this step"):
    """Run a chapter step if its inputs exist; a missing input or an error gives one line, never a traceback."""
    tables = [n for n in needs if n in TABLE_NAMES and T.get(n) is None]
    results = [n for n in needs if n not in TABLE_NAMES and R.get(n) is None]
    if tables:
        return skip(where, "needs the table(s) " + ", ".join(tables) + ", not in your export")
    if results:
        return skip(where, "needs the result(s) " + ", ".join(results) + " from the step above")
    try:
        return fn()
    except Exception as e:                      # a column missing in an unusual export, an empty table...
        skip(where, f"{type(e).__name__}: {e}")

# --- figure helpers --------------------------------------------------------------------------------------------
def style_ax(ax, grid=True, baseline=True):
    ax.set_facecolor("white")
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_visible(baseline)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(length=0, colors=TICK, labelsize=9)
    if grid:
        ax.yaxis.grid(True, color=GRID, linewidth=1)
        ax.set_axisbelow(True)
    return ax

def card(title, note=None, h=3.5, nrows=1, ncols=1, legend_rows=0, width_ratios=None, height_ratios=None, wspace=0.22, hspace=0.55, pad_top=0.0):
    """A figure styled like a dashboard card: dark title, grey note, white background, thin border.
    h = height of the plotting area in inches (3.5 default, 5 for .tall, 2.5 for .short); legend_rows reserves space
    for legend_below(); pad_top adds room under the note for panel titles or top axis labels. Returns (fig, ax) or (fig, [[ax, ...], ...]) for a grid of panels."""
    note_lines = textwrap.wrap(note, 128) if note else []
    top_in = 0.52 + 0.165 * len(note_lines) + pad_top
    bot_in = 0.42 + 0.26 * legend_rows
    H = h + top_in + bot_in
    fig = plt.figure(figsize=(8, H))
    fig.patch.set_facecolor("white"); fig.patch.set_edgecolor(BORDER); fig.patch.set_linewidth(1.5)
    fig.text(0.035, 1 - 0.13 / H, title, ha="left", va="top", color=INK, fontsize=10.5, fontweight="bold")
    for i, line in enumerate(note_lines):
        fig.text(0.035, 1 - (0.40 + 0.165 * i) / H, line, ha="left", va="top", color=TICK, fontsize=8.3)
    gs = fig.add_gridspec(nrows, ncols, left=0.075, right=0.975, top=1 - top_in / H, bottom=bot_in / H,
                          wspace=wspace, hspace=hspace, width_ratios=width_ratios, height_ratios=height_ratios)
    axes = [[style_ax(fig.add_subplot(gs[r, c])) for c in range(ncols)] for r in range(nrows)]
    fig._card = {"H": H, "bot_in": bot_in}
    return (fig, axes[0][0]) if nrows == ncols == 1 else (fig, axes)

def legend_below(fig, items, ncol=None):
    """The page's .legend: small grey labels with square swatches, under the chart. items = [(label, colour), ...]."""
    handles = [Patch(facecolor=c, edgecolor="none") for _l, c in items]
    labels = [l for l, _c in items]
    if ncol is None:
        ncol = 1 if sum(len(l) for l in labels) > 95 or len(labels) > 4 else len(labels)
        if len(labels) > 4:
            ncol = 2 if max(len(l) for l in labels) < 45 else 1
    fig.legend(handles, labels, loc="lower left", bbox_to_anchor=(0.035, 0.012), ncol=ncol, frameon=False, fontsize=8.3,
               labelcolor=TICK, handlelength=0.9, handleheight=0.9, handletextpad=0.5, columnspacing=1.6, borderaxespad=0)

def pct_axis(ax, axis="y", ticks=None):
    """Percent labels 0–100% (the page's d3 '.0%')."""
    a = ax.yaxis if axis == "y" else ax.xaxis
    a.set_major_formatter(FuncFormatter(lambda v, _p: fmtP(v)))
    if ticks is not None:
        a.set_ticks(ticks)

def count_axis(ax, axis="y", nticks=4):
    a = ax.yaxis if axis == "y" else ax.xaxis
    a.set_major_locator(MaxNLocator(nticks, integer=True))
    a.set_major_formatter(FuncFormatter(lambda v, _p: fmtN(int(v)) if float(v).is_integer() else fmtN(v)))

def year_ticks(ax, years, every=1, two_digit=False):
    ax.set_xticks(list(years))
    ax.set_xticklabels([(str(y)[2:] if two_digit else str(y)) if i % every == 0 else "" for i, y in enumerate(years)])

def strong(ax, x, y, s, **kw):
    """Emphasised label (the page's text.strong)."""
    kw.setdefault("color", INK); kw.setdefault("fontsize", 9)
    return ax.text(x, y, s, **kw)

def truncate(s, n=20):
    s = str(s)
    return s if len(s) <= n else s[:n - 1] + "…"

def done(fig, chart_id=None):
    """Show the finished card (chart_id = the id of the chart on the dashboard, for reference)."""
    plt.show()

## 0c. Build the analysis tables

The dashboard does not read the export tables directly; a build step first derives a few analysis tables from them,
and every chart is a calculation on one of these. This cell rebuilds them from *your* export, with the same columns:

| table | one row per… | built from |
|---|---|---|
| `events` | action you took (message, like, story reaction, save, comment, post, story), with Rome-time calendar fields | `messages`, `likes`, `story_likes`, `saved`, `comments`, `posts`, `stories` (+ the JSON messages for the domain of shared links) |
| `threads` | conversation | `threads` |
| `graph` | follow relationship, in either direction, with the year it started | `followers`, `following` |
| `feed` | video, post or ad the platform showed you in the retained window | `videos_watched`, `posts_viewed`, `ads_viewed` |
| `post_insights` | own post with creator insights | `insights_posts` |
| `advertisers` | business holding a match on you | `advertisers` |
| `ads` | ad shown (who posted it, for whom, with which hashtags) plus each organic item of the window | the JSON `ads_information/ads_and_topics/*` + `advertisers` |
| `convo` | message, with its reply latency and its burst ("session") | `messages`, `threads` |
| `taste` | post liked / saved / story reacted to / own post, with the author's network flag | `likes`, `saved`, `story_likes`, `posts`, `insights_posts`, `following` |
| `topics` | post liked or saved (with its lexicon topic), comment written (with its tone), feedback event | `likes`, `saved`, `comments`, `feedback` |

A table whose sources are missing is simply absent, and the chapters that need it say so.

In [ ]:
T = {}
TABLE_NAMES = ["events", "threads", "graph", "feed", "post_insights", "advertisers", "ads", "convo", "taste", "topics"]
EV_COLS = ["kind", "timestamp", "year", "quarter", "hour_local", "weekday_local", "actor", "actor_public", "thread", "is_me",
           "is_share", "n_words", "n_emoji", "has_question", "share_domain", "is_public", "date_local", "month_local", "week_local"]

def local_fields(ts):
    """Calendar fields in Rome time for a UTC datetime Series (the build script's local_fields)."""
    loc = ts.dt.tz_convert(TZ)
    naive = loc.dt.tz_localize(None)
    monday = naive.dt.normalize() - pd.to_timedelta(loc.dt.weekday, unit="D")
    return pd.DataFrame({"timestamp": ts.dt.strftime("%Y-%m-%dT%H:%M:%SZ"), "year": loc.dt.year.astype(int),
                         "quarter": loc.dt.year.astype(str) + "Q" + loc.dt.quarter.astype(str), "hour_local": loc.dt.hour.astype(int),
                         "weekday_local": loc.dt.weekday.astype(int), "date_local": loc.dt.strftime("%Y-%m-%d"),
                         "month_local": loc.dt.strftime("%Y-%m"), "week_local": monday.dt.strftime("%Y-%m-%d")})

def dated(df):
    """Rows with a valid timestamp, and their UTC datetimes."""
    ts = to_utc(df["timestamp"])
    ok = ts.notna().values
    return df[ok].reset_index(drop=True), ts[ok].reset_index(drop=True)

def event_block(kind, df, **cols):
    """One kind of event in the events table; cols are constants or functions of the (dated) source rows."""
    df, ts = dated(df)
    d = local_fields(ts)
    d["kind"] = kind
    defaults = dict(actor="", actor_public=False, thread="", is_me=False, is_share=False, n_words=0, n_emoji=0, has_question=False, share_domain="", is_public=False)
    for c, v in defaults.items():
        v = cols.get(c, v)
        d[c] = v(df).values if callable(v) else v
    return d[EV_COLS]

def share_domains():
    """(thread title, timestamp_ms, sender) -> domain of the shared link, from the JSON messages (the CSV has no domain)."""
    dom = {}
    if not JSON_DIR:
        return dom
    for f in glob.glob(os.path.join(JSON_DIR, "your_instagram_activity", "messages", "*", "*", "message_*.json")):
        try:
            d = json.load(open(f, encoding="utf-8"))
            for msg in d.get("messages", []):
                if "share" in msg:
                    dom[(d.get("title"), msg.get("timestamp_ms"), msg.get("sender_name"))] = (msg.get("share") or {}).get("domain") or "(no link)"
        except Exception:
            pass
    return dom

def build_events():
    parts = []
    m = load("messages")
    if have(m, "timestamp", "thread", "sender", "is_me"):
        m, ts = dated(m)
        dom = share_domains()
        ms = ((ts - pd.Timestamp(0, tz="UTC")) // pd.Timedelta(milliseconds=1)).astype("int64")
        m["share_domain"] = [dom.get(k, "") for k in zip(m["thread"], ms, m["sender"])]
        m["is_share"] = m["attachment"].astype(str).eq("share") if "attachment" in m else False
        m.loc[m["is_share"] & (m["share_domain"] == ""), "share_domain"] = "(no link)"
        m.loc[~m["is_share"], "share_domain"] = ""
        for c in ["n_words", "n_emoji"]:
            m[c] = pd.to_numeric(m[c], errors="coerce").fillna(0).astype(int) if c in m else 0
        m["has_question"] = flag(m["has_question"]) if "has_question" in m else False
        parts.append(event_block("message", m, actor=lambda d: d["sender"], thread=lambda d: d["thread"], is_me=lambda d: flag(d["is_me"]),
                                 is_share=lambda d: d["is_share"], n_words=lambda d: d["n_words"], n_emoji=lambda d: d["n_emoji"],
                                 has_question=lambda d: d["has_question"], share_domain=lambda d: d["share_domain"]))
    lk = load("likes")
    if have(lk, "timestamp", "author", "in_network"):
        lp = (lk[lk["kind"].astype(str) == "post"] if "kind" in lk else lk).reset_index(drop=True)   # likes on comments are not events on the page
        if len(lp):
            parts.append(event_block("like", lp, actor=lambda d: d["author"], actor_public=lambda d: ~flag(d["in_network"]), is_public=True))
    for table, kind in [("story_likes", "story_like"), ("saved", "save")]:
        df = load(table)
        if have(df, "timestamp", "author", "in_network"):
            parts.append(event_block(kind, df, actor=lambda d: d["author"], actor_public=lambda d: ~flag(d["in_network"])))
    cm = load("comments")
    if have(cm, "timestamp"):
        parts.append(event_block("comment", cm, actor=lambda d: d["on_post_by"].fillna("") if "on_post_by" in d else pd.Series("", index=d.index), is_public=True))
    po = load("posts")
    if have(po, "timestamp"):
        parts.append(event_block("post", po, is_public=True))
    st = load("stories")
    if have(st, "timestamp"):
        parts.append(event_block("story", st))
    return pd.concat(parts, ignore_index=True) if parts else None

def build_threads():
    t = load("threads")
    if not have(t, "thread", "thread_type", "n_messages", "n_from_me", "first_message", "last_message"):
        return None
    t = t.copy()
    t["first_message"] = to_utc(t["first_message"]).dt.strftime("%Y-%m-%d")
    t["last_message"] = to_utc(t["last_message"]).dt.strftime("%Y-%m-%d")
    t["share_me"] = (t["n_from_me"] / t["n_messages"]).round(3)
    t = t.sort_values(["n_messages", "thread"], ascending=[False, True]).reset_index(drop=True)
    cols = [c for c in ["thread", "thread_type", "n_participants", "n_messages", "n_from_me", "first_message", "last_message", "share_me"] if c in t]
    return t[cols]

def build_graph():
    fo, fi = load("followers"), load("following")
    if not (have(fo, "user", "timestamp") and have(fi, "user", "timestamp")):
        return None
    both = set(fo["user"]) & set(fi["user"])
    g = pd.concat([fo.assign(direction="follower"), fi.assign(direction="following")], ignore_index=True)
    g, ts = dated(g)
    g["year"] = ts.dt.tz_convert(TZ).dt.year.astype(int)
    g["mutual"] = g["user"].isin(both)
    g["timestamp"] = ts.dt.strftime("%Y-%m-%dT%H:%M:%S+00:00")
    return g[["user", "direction", "year", "mutual", "timestamp"]]

def build_feed():
    def block(df, kind, source=None):
        df, ts = dated(df)
        loc = ts.dt.tz_convert(TZ)
        inn = flag(df["in_network"]) if source is None else pd.Series(False, index=df.index)
        return pd.DataFrame({"kind": kind, "day": loc.dt.strftime("%Y-%m-%d"), "hour_local": loc.dt.hour.astype(int), "account": df["author"],
                             "in_network": inn, "source": source if source else inn.map({True: "followed", False: "not followed"}),
                             "n_words_caption": pd.to_numeric(df["n_words_caption"], errors="coerce").fillna(0).astype(int) if "n_words_caption" in df else 0,
                             "hashtags": df["hashtags"].fillna("") if "hashtags" in df else ""})
    parts = []
    for table, kind, source in [("videos_watched", "video", None), ("posts_viewed", "post", None), ("ads_viewed", "ad", "ad")]:
        df = load(table)
        if have(df, "timestamp", "author") and (source or "in_network" in df):
            parts.append(block(df, kind, source))
    return pd.concat(parts, ignore_index=True) if parts else None

def build_post_insights():
    ip = load("insights_posts")
    if not have(ip, "timestamp", "likes"):
        return None
    ip, ts = dated(ip)
    num = lambda c: pd.to_numeric(ip[c], errors="coerce").astype("Int64") if c in ip else pd.array([None] * len(ip), dtype="Int64")
    pi = pd.DataFrame({"timestamp": ts.dt.strftime("%Y-%m-%d"), "likes": num("likes"), "comments": num("comments"), "saves": num("saves")})
    return pi.sort_values("timestamp", kind="stable").reset_index(drop=True)

def build_advertisers():
    a = load("advertisers")
    return a if have(a, "advertiser") else None

# --- ads (chapter 10): from the JSON tree, where the ad records keep the brand partner and the posting account
DISC = {"adv", "ad", "ads", "sponsored", "sponsorizzato", "pubblicita", "gifted", "paidpartnership", "partnership", "collab", "collaboration",
        "supplied", "invited", "advertising", "advertisement", "adv_", "suppliedforreview"}
DISC_RE = re.compile(r"paid partnership|partnership retribuita|in collaborazione con|sponsored|sponsorizzat|pubblicit", re.I)
TAG_RE = re.compile(r"#([\wÀ-ɏ]+)")
META_RE = re.compile(r"\[(\d+) words, (\d+) emoji\]")      # organic captions are reduced to '#tags @mentions [N words, M emoji]'

def lv_fields(item):
    """Flatten an export record's label_values into {label: value}; nested blocks become lists of dicts."""
    out = {}
    for e in item.get("label_values") or []:
        if not isinstance(e, dict):
            continue
        name = e.get("label") if isinstance(e.get("label"), str) else e.get("title") if "dict" in e else None
        if name is None:
            continue
        if "value" in e:
            out[name] = e["value"]
        elif "timestamp_value" in e:
            out[name] = e["timestamp_value"]
        elif "vec" in e:
            out[name] = [x.get("value") if isinstance(x, dict) else x for x in e["vec"]]
        elif "dict" in e:
            out[name] = [lv_fields({"label_values": sub.get("dict", [])}) for sub in e["dict"] if isinstance(sub, dict)]
    return out

def build_ads():
    base = os.path.join(JSON_DIR or "", "ads_information", "ads_and_topics")
    if not JSON_DIR or not os.path.exists(os.path.join(base, "ads_viewed.json")):
        return None
    def load_json(name):
        p = os.path.join(base, name + ".json")
        if not os.path.exists(p):
            return []
        d = json.load(open(p, encoding="utf-8"))
        return d if isinstance(d, list) else d[list(d.keys())[0]]
    first = lambda block: block[0] if isinstance(block, list) and block else {}
    norm = lambda s: re.sub(r"[^a-z0-9]", "", (s or "").lower())
    def n_words(cap):
        mm = META_RE.search(cap or "")
        return int(mm.group(1)) if mm else len(re.findall(r"\S+", cap or ""))
    def tags_of(f):
        cap = f.get("Caption") or ""
        tags = [h.get("Name", "") for h in (f.get("Hashtags") or []) if isinstance(h, dict) and h.get("Name")]
        alltags = [t.lower() for t in tags + TAG_RE.findall(cap)]
        return alltags, cap, any(norm(t) in DISC for t in alltags) or bool(DISC_RE.search(cap))
    def row(kind, it, f, **extra):
        ts = pd.Timestamp(it["timestamp"], unit="s", tz="UTC")
        alltags, cap, disc = tags_of(f)
        o = first(f.get("Owner"))
        r = dict(kind=kind, timestamp=ts.strftime("%Y-%m-%dT%H:%M:%SZ"), day_local=ts.tz_convert(TZ).strftime("%Y-%m-%d"), hour_local=ts.tz_convert(TZ).hour,
                 owner=(o.get("Username") or "").strip(), owner_name="", brand_partner="", brand_partner_name="", advertiser="", same_account=True,
                 has_disclosure=disc, n_hashtags=len(set(alltags)), n_words_caption=n_words(cap), hashtags=" ".join(dict.fromkeys(alltags)),
                 appears_organic=False, in_videos_watched=False, in_customer_lists="")
        r.update(extra)
        return r
    ads_raw = [(it, lv_fields(it)) for it in load_json("ads_viewed")]
    if not ads_raw:
        return None
    adkeys = {(it["timestamp"], (first(f.get("Owner")).get("Username") or "").strip().lower()) for it, f in ads_raw}
    organic, dupl, organic_authors = [], set(), set()
    for name, kind in [("videos_watched", "video"), ("posts_viewed", "post")]:
        for it in load_json(name):
            f = lv_fields(it)
            u = (first(f.get("Owner")).get("Username") or "").strip().lower()
            if (it["timestamp"], u) in adkeys:          # copies of ads listed among watched videos / viewed posts
                dupl.add((it["timestamp"], u)); continue
            organic.append((kind, it, f))
            if u:
                organic_authors.add(u)
    adv = list(T["advertisers"]["advertiser"].astype(str)) if T.get("advertisers") is not None else []
    adv_norm = [(a, norm(a), norm(a.split()[0]) if a.split() else "") for a in adv]
    def in_lists(*names):      # loose match of the advertiser with the customer-list holders: whole name, or first word of >= 4 letters
        cands = [norm(c) for c in names if norm(c)]
        for a, an, af in adv_norm:
            for c in cands:
                if c == an or (len(af) >= 4 and (c.startswith(af) or (len(c) >= 4 and af.startswith(c)))):
                    return a
        return ""
    rows = []
    for it, f in ads_raw:
        o, b = first(f.get("Owner")), first(f.get("Brand partner"))
        owner, owner_name = (o.get("Username") or "").strip(), (o.get("Name") or "").strip()
        bp, bp_name = (b.get("Username") or "").strip(), (b.get("Name") or "").strip()
        same = bp == "" or bp.lower() == owner.lower()
        rows.append(row("ad", it, f, owner_name=owner_name, brand_partner=bp, brand_partner_name=bp_name, advertiser=bp if bp else owner, same_account=same,
                        appears_organic=(owner.lower() in organic_authors) or (bp != "" and bp.lower() in organic_authors),
                        in_videos_watched=(it["timestamp"], owner.lower()) in dupl, in_customer_lists=in_lists(bp_name or owner_name, bp or owner)))
    for kind, it, f in organic:
        rows.append(row(kind, it, f))
    return pd.DataFrame(rows)

def build_convo():
    m, t = load("messages"), load("threads")
    if not (have(m, "thread", "sender", "is_me", "timestamp") and have(t, "thread", "thread_type")):
        return None
    m, ts = dated(m)
    m["ts"] = ts
    m = m.merge(t[["thread", "thread_type"]].drop_duplicates("thread"), on="thread", how="left")
    m = m.sort_values(["thread", "ts"], kind="stable").reset_index(drop=True)
    loc = m["ts"].dt.tz_convert(TZ)
    m["year"] = loc.dt.year.astype(int); m["date_local"] = loc.dt.strftime("%Y-%m-%d"); m["hour_local"] = loc.dt.hour; m["weekday_local"] = loc.dt.weekday
    m["is_me"] = flag(m["is_me"])
    m["is_share"] = m["attachment"].astype(str).eq("share") if "attachment" in m else False
    for c in ["has_question", "has_link"]:
        m[c] = flag(m[c]) if c in m else False
    g = m.groupby("thread")
    prev_ts, prev_sender = g["ts"].shift(1), g["sender"].shift(1)
    gap_min = (m["ts"] - prev_ts).dt.total_seconds() / 60
    m["gap_min"] = gap_min.round(1)
    diff = prev_sender.notna() & (prev_sender != m["sender"])
    m["reply_latency_min"] = gap_min.clip(upper=7 * 24 * 60).where(diff).round(1)       # reply latency, capped at 7 days
    new_sess = prev_ts.isna() | (gap_min >= 30)                                          # a burst ends after 30 quiet minutes
    m["session_seq"] = new_sess.groupby(m["thread"]).cumsum().astype(int)
    m["session_id"] = m["thread"].astype(str) + "#" + m["session_seq"].astype(str)
    m["session_pos"] = m.groupby("session_id").cumcount() + 1
    m["session_size"] = m.groupby("session_id")["session_pos"].transform("max")
    m["session_senders"] = m.groupby("session_id")["sender"].transform("nunique")
    m["thread_first_year"] = m.groupby("thread")["year"].transform("min")
    m["thread_last_year"] = m.groupby("thread")["year"].transform("max")
    m["timestamp"] = m["ts"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    cols = ["thread", "thread_type", "sender", "is_me", "timestamp", "year", "date_local", "hour_local", "weekday_local", "is_share", "has_question", "has_link",
            "gap_min", "reply_latency_min", "session_id", "session_seq", "session_pos", "session_size", "session_senders", "thread_first_year", "thread_last_year"]
    return m[cols]

def build_taste():
    COLS = ["kind", "timestamp", "year", "author", "in_network", "pre_follow", "followed_at", "is_self", "n_words_caption", "n_hashtags",
            "n_mentions", "n_media", "likes_received", "comments_received", "saves_received"]
    n_tags = lambda s: s.fillna("").astype(str).str.split().str.len().astype(int)
    fol = load("following")
    followed_at = pd.Series(dtype=object)
    if have(fol, "user", "timestamp"):
        fol = fol.assign(followed_at=to_utc(fol["timestamp"]).dt.strftime("%Y-%m-%d"))
        followed_at = fol.drop_duplicates("user").set_index("user")["followed_at"]
    def reaction(df, kind):
        df, ts = dated(df)
        d = pd.DataFrame({"kind": kind, "timestamp": ts.dt.strftime("%Y-%m-%dT%H:%M:%SZ"), "year": ts.dt.year.astype(int), "author": df["author"], "in_network": flag(df["in_network"])})
        d["followed_at"] = d["author"].map(followed_at)
        d["pre_follow"] = d["followed_at"].notna() & (d["timestamp"].str[:10] < d["followed_at"].fillna(""))
        d["is_self"] = d["author"].astype(str).eq("me")
        d["n_words_caption"] = pd.to_numeric(df["n_words_caption"], errors="coerce").fillna(0).astype(int) if "n_words_caption" in df else 0
        d["n_hashtags"] = n_tags(df["hashtags"]) if "hashtags" in df else 0
        for c in ["n_mentions", "n_media", "likes_received", "comments_received", "saves_received"]:
            d[c] = pd.array([None] * len(d), dtype="Int64")
        return d[COLS]
    parts = []
    lk = load("likes")
    if have(lk, "timestamp", "author", "in_network"):
        lp = lk[lk["kind"].astype(str) == "post"] if "kind" in lk else lk
        parts.append(reaction(lp.reset_index(drop=True), "like"))
    for table, kind in [("saved", "save"), ("story_likes", "story_like")]:
        df = load(table)
        if have(df, "timestamp", "author", "in_network"):
            parts.append(reaction(df, kind))
    po = load("posts")
    if have(po, "timestamp"):
        po, pts = dated(po)
        num = lambda s: pd.to_numeric(s, errors="coerce").astype("Int64")
        p = pd.DataFrame({"kind": "post", "timestamp": pts.dt.strftime("%Y-%m-%dT%H:%M:%SZ"), "year": pts.dt.year.astype(int), "author": "me", "in_network": True,
                          "pre_follow": False, "followed_at": None, "is_self": True,
                          "n_words_caption": pd.to_numeric(po["n_words"], errors="coerce").fillna(0).astype(int) if "n_words" in po else 0,
                          "n_hashtags": n_tags(po["hashtags"]) if "hashtags" in po else 0,
                          "n_mentions": num(po["n_mentions"]) if "n_mentions" in po else pd.array([None] * len(po), dtype="Int64"),
                          "n_media": num(po["n_media"]) if "n_media" in po else pd.array([None] * len(po), dtype="Int64")})
        ins = load("insights_posts")
        if have(ins, "timestamp", "likes"):
            ins, its = dated(ins)
            ins = pd.DataFrame({"key": its.dt.strftime("%Y-%m-%dT%H:%M:%SZ"), "likes_received": num(ins["likes"]),
                                "comments_received": num(ins["comments"]) if "comments" in ins else pd.array([None] * len(ins), dtype="Int64"),
                                "saves_received": num(ins["saves"]) if "saves" in ins else pd.array([None] * len(ins), dtype="Int64")}).drop_duplicates("key")
            p = p.merge(ins, left_on="timestamp", right_on="key", how="left").drop(columns="key")
        else:
            for c in ["likes_received", "comments_received", "saves_received"]:
                p[c] = pd.array([None] * len(p), dtype="Int64")
        parts.append(p[COLS])
    if not parts:
        return None
    return pd.concat(parts, ignore_index=True).sort_values(["kind", "timestamp"], kind="stable").reset_index(drop=True)

def build_topics():
    COLS = ["kind", "timestamp", "year", "in_network", "n_words", "n_hashtags", "topic", "tone", "feedback"]
    n_tags = lambda s: s.fillna("").astype(str).str.split().str.len().astype(int)
    parts, has_labels = [], False
    def base(df, kind):
        df, ts = dated(df)
        d = pd.DataFrame({"kind": kind, "timestamp": ts.dt.strftime("%Y-%m-%dT%H:%M:%S+00:00"), "year": ts.dt.year.astype(int)})
        d["in_network"] = flag(df["in_network"]) if "in_network" in df else df["on_post_by"].astype(str).str.match(r"^(me|user_\d+)$") if "on_post_by" in df else True
        nw = df["n_words_caption"] if "n_words_caption" in df else df["n_words"] if "n_words" in df else pd.Series(0, index=df.index)
        d["n_words"] = pd.to_numeric(nw, errors="coerce").fillna(0).astype(int)
        d["n_hashtags"] = n_tags(df["hashtags"]) if "hashtags" in df else 0
        d["topic"] = df["topic"].where(df["topic"].notna(), None).values if "topic" in df else None
        d["tone"] = df["tone"].values if "tone" in df else None
        d["feedback"] = df["feedback"].values if "feedback" in df else None
        return d[COLS], ("topic" in df and df["topic"].notna().any()) or ("tone" in df and df["tone"].notna().any())
    lk = load("likes")
    if have(lk, "timestamp"):
        lp = lk[lk["kind"].astype(str) == "post"] if "kind" in lk else lk
        d, h = base(lp.reset_index(drop=True), "like"); parts.append(d); has_labels |= h
    for table, kind in [("saved", "save"), ("comments", "comment"), ("feedback", "feedback")]:
        df = load(table)
        if have(df, "timestamp"):
            d, h = base(df, kind); parts.append(d); has_labels |= h
    if not parts or not has_labels:
        return None          # no topic/tone columns: the topic step of notebook 02 was not run
    return pd.concat(parts, ignore_index=True).sort_values("timestamp", kind="stable").reset_index(drop=True)

builders = {"events": build_events, "threads": build_threads, "graph": build_graph, "feed": build_feed, "post_insights": build_post_insights,
            "advertisers": build_advertisers, "ads": build_ads, "convo": build_convo, "taste": build_taste, "topics": build_topics}
if DATA:
    for name, fn in builders.items():
        try:
            T[name] = fn()
        except Exception as e:
            print(f"  ({name}: could not be built - {type(e).__name__}: {e})")
            T[name] = None
    summary = pd.DataFrame([(n, 0 if T.get(n) is None else len(T[n]), "" if T.get(n) is None else ", ".join(map(str, T[n].columns[:7])) + (" ..." if T[n].shape[1] > 7 else ""))
                            for n in TABLE_NAMES], columns=["table", "rows", "columns"]).set_index("table")
    display(summary)
    if T.get("events") is not None:
        print("events by kind:", T["events"]["kind"].value_counts().to_dict())

### The headline numbers

The dashboard opens with six tiles — the whole history in six counts — and one sentence on the period covered. The
same here, from your tables (calcs `totals` and `span`).

In [ ]:
def calc_totals():
    ev = T["events"]; n = ev["kind"].value_counts()
    posts = sorted(ev.loc[ev["kind"] == "post", "timestamp"])
    R["totals"] = dict(messages=int(n.get("message", 0)), likes=int(n.get("like", 0)), story_likes=int(n.get("story_like", 0)), saves=int(n.get("save", 0)),
                       posts=int(n.get("post", 0)), comments=int(n.get("comment", 0)),
                       threads=len(T["threads"]) if T.get("threads") is not None else None,
                       followers=int((T["graph"]["direction"] == "follower").sum()) if T.get("graph") is not None else None,
                       last_post=posts[-1][:7] if posts else "—")
    t = sorted(ev["timestamp"]); y0, y1 = t[0][:4], t[-1][:4]
    R["span"] = dict(first=t[0][:10], last=t[-1][:10], y0=y0, y1=y1, last_post_year=posts[-1][:4] if posts else "—")
    tiles = [("direct messages, " + y0 + "–" + y1, fmtN(R["totals"]["messages"])), ("conversations", fmtN(R["totals"]["threads"])), ("posts liked", fmtN(R["totals"]["likes"])),
             ("story reactions", fmtN(R["totals"]["story_likes"])), ("posts saved", fmtN(R["totals"]["saves"])),
             ("posts published, last one " + (R["totals"]["last_post"] or "—"), fmtN(R["totals"]["posts"]))]
    display(pd.DataFrame(tiles, columns=["", "n"]).set_index("").T)
    print(f"Your dated actions run from {R['span']['first']} to {R['span']['last']}; the last post is from {R['span']['last_post_year']}.")
run(calc_totals, needs=["events"], where="Headline numbers")

## Explore — Any action, any period

Count what you like over the period you like. Your own activity runs from 2018 to 2026; what the platform showed you exists only for the five days the export retains, so for the feed only day, weekday and hour carry information.

Set `WHAT` to one of `activity` (all your own actions, stacked by type), `message`, `like`, `story_like`, `save`,
`comment`, `post`, or `feed` (the videos, posts and ads the platform showed you), and `GRAIN` to `year`, `quarter`,
`month`, `week`, `day`, `weekday` or `hour`, then re-run the cell. This is the page's explorer (calc `by_period`): the
stacked bars become a stacked area when there are more than 60 periods, as on the page.

In [ ]:
WHAT = "activity"     # activity | message | like | story_like | save | comment | post | feed
GRAIN = "year"        # year | quarter | month | week | day | weekday | hour

SERIES_ACT = ["post", "comment", "like", "story_like", "save", "message"]
SERIES_FEED = ["feed_video", "feed_post", "feed_ad"]
SERIES_LABEL = dict(KIND_LABEL, feed_video="Videos shown", feed_post="Posts shown", feed_ad="Ads shown")

def by_period(what, grain):
    """Counts per (series, period) - the page's by_period calc restricted to one WHAT and one GRAIN."""
    col = {"year": "year", "quarter": "quarter", "month": "month_local", "week": "week_local", "day": "date_local", "weekday": "weekday_local", "hour": "hour_local"}[grain]
    out = {}
    if what == "feed":
        f = T.get("feed")
        if f is None:
            return {}
        day = pd.to_datetime(f["day"])
        per = {"year": day.dt.strftime("%Y"), "quarter": day.dt.year.astype(str) + "Q" + day.dt.quarter.astype(str), "month": day.dt.strftime("%Y-%m"), "week": f["day"],
               "day": f["day"], "weekday": day.dt.weekday.astype(str), "hour": f["hour_local"].astype(str)}[grain]
        for k, n in f.assign(series="feed_" + f["kind"].astype(str), period=per.astype(str)).groupby(["series", "period"]).size().items():
            out[k] = int(n)
    else:
        ev = T.get("events")
        if ev is None:
            return {}
        ev = ev[ev["kind"] != "story"]
        if what != "activity":
            ev = ev[ev["kind"] == what]
        for k, n in ev.assign(period=ev[col].astype(str)).groupby(["kind", "period"]).size().items():
            out[k] = int(n)
    return out

def draw_explore():
    series = SERIES_ACT if WHAT == "activity" else SERIES_FEED if WHAT == "feed" else [WHAT]
    colors = C[:6] if WHAT == "activity" else [C[0], C[2], C[3]] if WHAT == "feed" else [C[1]]
    counts = by_period(WHAT, GRAIN)
    ps = sorted({p for (s, p) in counts if s in series})
    if GRAIN == "weekday":
        periods = [str(i) for i in range(7)]
    elif GRAIN == "hour":
        periods = [str(i) for i in range(24)]
    elif not ps:
        return skip("Explore", f"no {WHAT} events")
    elif GRAIN == "year":
        periods = [str(y) for y in range(int(ps[0]), int(ps[-1]) + 1)]
    elif GRAIN == "quarter":
        periods = [f"{p.year}Q{p.quarter}" for p in pd.period_range(ps[0], ps[-1], freq="Q")]
    elif GRAIN == "month":
        periods = [p.strftime("%Y-%m") for p in pd.period_range(ps[0], ps[-1], freq="M")]
    else:
        step = 7 if GRAIN == "week" else 1
        periods = [d.strftime("%Y-%m-%d") for d in pd.date_range(ps[0], ps[-1], freq=f"{step}D")]
    data = pd.DataFrame({s: [counts.get((s, p), 0) for p in periods] for s in series}, index=periods)
    def label(p):
        if GRAIN == "weekday": return WD[int(p)]
        if GRAIN == "hour": return f"{int(p):02d}:00"
        if GRAIN == "month": d = pd.Timestamp(p + "-01"); return f"{d:%b %y}"
        if GRAIN in ("year", "quarter"): return p
        d = pd.Timestamp(p); return f"{d.day} {d:%b}"
    dense = len(periods) > 60
    fig, ax = card("Any action, any period", f"{'All of my activity, by type' if WHAT == 'activity' else SERIES_LABEL.get(WHAT, 'Feed window: videos, posts and ads shown')} per {GRAIN}.", h=5, legend_rows=1 if len(series) <= 3 else 2)
    x = range(len(periods)); bottom = pd.Series(0.0, index=data.index)
    for s, col in zip(series, colors):
        if dense:
            ax.fill_between(list(x), bottom.values, (bottom + data[s]).values, color=col, alpha=0.85, linewidth=0)
        else:
            ax.bar(list(x), data[s].values, bottom=bottom.values, color=col, width=0.8)
        bottom = bottom + data[s]
    every = max(1, math.ceil(len(periods) / (8 if dense else 12)))
    ax.set_xticks([i for i in x if i % every == 0]); ax.set_xticklabels([label(p) for i, p in enumerate(periods) if i % every == 0])
    ax.set_xlim(-0.6, len(periods) - 0.4); ax.set_ylim(0, max(1, bottom.max()) * 1.04)
    count_axis(ax, nticks=5)
    legend_below(fig, [(SERIES_LABEL[s], c) for s, c in zip(series, colors)])
    done(fig, "c-explore")
run(draw_explore, needs=["events"], where="Explore")

## Chapter 1 — Posting stopped; consuming moved out of sight

**The economics question.** Platforms earn from attention, and attention changes form over a user's life: a typical
account posts a lot in its first years, then settles into private messaging and passive consumption. Every public
action (a post, a comment, a like) produces content or signals for *other users*; every private one (a message, a story
reaction, a save) produces data for the *platform* only. The dashboard's first chapter measures that shift from the
export: actions per year by type (calc `by_year_kind`, `peaks`) and the share of activity visible to others
(`public_share`).

In [ ]:
KINDS = ["post", "comment", "like", "story_like", "save", "message"]
PUBLIC = ["post", "comment", "like"]

def calc_arc():
    ev = T["events"]
    years = sorted(ev["year"].unique())
    byk = ev[ev["kind"].isin(KINDS)].groupby(["kind", "year"]).size()
    rows = [dict(kind=k, year=int(y), n=int(byk.get((k, y), 0))) for k in KINDS for y in years]
    R["by_year_kind"] = pd.DataFrame(rows)
    b = R["by_year_kind"]
    def peak(k):
        d = b[b["kind"] == k]
        return int(d.loc[d["n"].idxmax(), "year"]) if len(d) and d["n"].max() > 0 else None
    pub = b[b["kind"].isin(PUBLIC)].groupby("year")["n"].sum()
    sl = b[(b["kind"] == "story_like") & (b["n"] > 0)]
    R["peaks"] = dict(public_peak_year=int(pub.idxmax()) if len(pub) and pub.max() > 0 else None, save_peak_year=peak("save"), message_peak_year=peak("message"),
                      story_like_first=int(sl["year"].min()) if len(sl) else None)
    g = ev.groupby("year")
    ps = pd.DataFrame({"year": years, "n": g.size().values, "n_public": g["is_public"].apply(lambda s: int(flag(s).sum())).values})
    ps["share_public"] = ps["n_public"] / ps["n"]
    R["public_share"] = ps
    say("Every public action \u2014 publishing, commenting, liking \u2014 peaks around [peaks.public_peak_year] and then fades; the last post is from [totals.last_post]. The private actions keep growing: story reactions start in [peaks.story_like_first], saves peak in [peaks.save_peak_year], messages in [peaks.message_peak_year]. By [public_share.year@-1], only [public_share.share_public@-1] of all recorded activity is visible to other users; in [public_share.year@0] it was [public_share.share_public@0].")
run(calc_arc, needs=["events"], where="Chapter 1")

In [ ]:
def draw_arc():
    b = R["by_year_kind"]; years = sorted(b["year"].unique())
    fig, axes = card("Actions per year, by type", "Each panel has its own scale; the bars are counts of one kind of action per calendar year.", h=4.6, nrows=2, ncols=3, wspace=0.18, hspace=0.5, pad_top=0.2)
    for i, kind in enumerate(KINDS):
        ax = axes[i // 3][i % 3]
        d = b[b["kind"] == kind].set_index("year")["n"].reindex(years, fill_value=0)
        ax.bar(range(len(years)), d.values, color=C[0] if kind in PUBLIC else C[1], width=0.75)
        ax.yaxis.grid(False); ax.set_yticks([])
        ax.text(0, 1.04, KIND_LABEL[kind], transform=ax.transAxes, color=INK, fontsize=8)
        ax.text(1, 1.04, "max " + fmtN(int(d.max())), transform=ax.transAxes, color=TICK, fontsize=7.5, ha="right")
        ticks = [j for j in range(len(years)) if j == 0 or j == len(years) - 1 or (len(years) > 6 and j % 4 == 0)]
        ax.set_xticks(ticks); ax.set_xticklabels([str(years[j]) for j in ticks]); ax.set_xlim(-0.6, len(years) - 0.4)
        ax.set_ylim(0, max(1, d.max()) * 1.05)
    done(fig, "c-arc")
run(draw_arc, needs=["by_year_kind"], where="Chapter 1 chart")

def draw_public():
    d = R["public_share"]
    fig, ax = card("Share of activity visible to others", "Posts, comments and likes are visible on the feed; messages, story reactions and saves are not.", h=2.5)
    ax.plot(d["year"], d["share_public"], color=C[0], linewidth=2, marker="o", markersize=5, solid_capstyle="round")
    ax.set_ylim(0, 1); pct_axis(ax, ticks=[0, 0.5, 1]); ax.spines["bottom"].set_visible(False)
    year_ticks(ax, d["year"]); ax.set_xlim(d["year"].min() - 0.4, d["year"].max() + 0.9)
    last = d.iloc[-1]; strong(ax, last["year"] + 0.15, last["share_public"], fmtP(last["share_public"]), va="center")
    done(fig, "c-public")
run(draw_public, needs=["public_share"], where="Chapter 1 chart")

## Chapter 2 — The inbox became a distribution channel

**The economics question.** A direct message used to be the one place on the platform the recommender did not reach.
When a growing share of messages are *shares* — a post or reel forwarded to a friend — the inbox becomes a distribution
channel for the platform's own content, and every share is a signal of what spreads. The chapter counts, per year,
messages that carry a shared post, reel or link against typed text, the share of those links that point back to
Instagram, and the emoji per message as a rough index of how much of the conversation is still conversation (calcs
`share_by_year`, `share_tot`).

In [ ]:
def calc_dms():
    m = T["events"][T["events"]["kind"] == "message"]
    if not len(m):
        return skip("Chapter 2", "no messages")
    g = m.groupby("year")
    d = pd.DataFrame({"year": sorted(m["year"].unique())})
    d["n"] = g.size().values; d["shares"] = g["is_share"].apply(lambda s: int(flag(s).sum())).values
    d["text"] = d["n"] - d["shares"]; d["share_pct"] = d["shares"] / d["n"]
    d["emoji"] = (g["n_emoji"].sum().values / d["n"]).round(2)
    R["share_by_year"] = d
    sh = m[flag(m["is_share"])]
    R["share_tot"] = dict(n=len(m), shares=len(sh), share_pct=len(sh) / len(m), ig_share_pct=(sh["share_domain"].astype(str) == "www.instagram.com").sum() / len(sh) if len(sh) else float("nan"))
    say("A third of all messages ([share_tot.share_pct]) are not text but a shared post or reel, and [share_tot.ig_share_pct] of those shares point back to Instagram. In [share_by_year.year@2] shares were [share_by_year.share_pct@2] of messages; in [share_by_year.year@-1] they are [share_by_year.share_pct@-1]. Emoji per message fell from [share_by_year.emoji@3] to [share_by_year.emoji@-1] over the same years: the conversation is increasingly made of the platform's own content.")
run(calc_dms, needs=["events"], where="Chapter 2")

def stacked_years(d, series, labels, colors, title, note, chart_id, h=3.5):
    """The page's stackedYears: stacked bars per year, 4 grid ticks, legend below."""
    fig, ax = card(title, note, h=h, legend_rows=1)
    bottom = pd.Series(0.0, index=d.index)
    for s, c in zip(series, colors):
        ax.bar(d["year"], d[s], bottom=bottom, color=c, width=0.75); bottom = bottom + d[s]
    year_ticks(ax, d["year"]); count_axis(ax); ax.spines["bottom"].set_visible(False); ax.set_ylim(0, bottom.max() * 1.05)
    legend_below(fig, list(zip(labels, colors)))
    done(fig, chart_id)
run(lambda: stacked_years(R["share_by_year"], ["text", "shares"], ["Typed text", "Shared post, reel or link"], [MUTED, C[1]], "Messages per year: text versus shared content", "A share is a message carrying a post, reel or link rather than typed text.", "c-shares"),
    needs=["share_by_year"], where="Chapter 2 chart")

## Chapter 3 — A handful of conversations carry everything

**The economics question.** Messaging is where the platform's *network effect* lives: Instagram is hard to leave not
because of the feed but because the people you talk to are there — and those people are few. The chapter ranks your
conversations by size (calc `conc`: share of all messages in the three and ten largest threads, the median thread, the
span of the largest) and asks who does the talking in direct threads (`recip`, `recip_hist`: the share of each thread's
messages that you sent).

In [ ]:
def calc_threads():
    t = T["threads"]
    tt = t[t["n_messages"] > 0].sort_values(["n_messages", "thread"], ascending=[False, True], kind="stable").reset_index(drop=True)
    tot = tt["n_messages"].sum()
    sizes = sorted(tt["n_messages"])
    yrs = (pd.Timestamp(tt.iloc[0]["last_message"]) - pd.Timestamp(tt.iloc[0]["first_message"])).days / 365.25
    R["conc"] = dict(n_threads=len(tt), top3=tt["n_messages"].head(3).sum() / tot, top10=tt["n_messages"].head(10).sum() / tot,
                     median=int(sizes[len(sizes) // 2]), top_years=int(math.floor(yrs + 0.5)))
    R["top_threads"] = tt.head(12).rename(columns={"thread_type": "type", "n_messages": "n", "n_from_me": "mine", "first_message": "first", "last_message": "last"})[["thread", "type", "n", "mine", "first", "last"]]
    d = t[(t["thread_type"] == "direct") & (t["n_messages"] > 0)]
    x = d["n_from_me"] / d["n_messages"]
    R["recip"] = dict(n_direct=len(d), n_under30=int((x < 0.3).sum()), n_over70=int((x > 0.7).sum()))
    bins = (x * 10).apply(math.floor).clip(upper=9).value_counts().reindex(range(10), fill_value=0)
    R["recip_hist"] = pd.DataFrame({"lo": [i / 10 for i in range(10)], "hi": [(i + 1) / 10 for i in range(10)], "n": bins.values.astype(int)})
    say("Of [conc.n_threads] conversations, the three largest hold [conc.top3] of all messages and the ten largest [conc.top10]; the median conversation has [conc.median] messages. The largest one has run for [conc.top_years] years. In most one-to-one threads this account is the quieter side: in [recip.n_under30] of [recip.n_direct] direct threads it sent fewer than 30% of the messages, in only [recip.n_over70] more than 70%.")
run(calc_threads, needs=["threads"], where="Chapter 3")

In [ ]:
def draw_threads():
    d = R["top_threads"]; y = range(len(d))
    fig, ax = card("The twelve largest conversations", "Messages per thread; the darker part is what this account sent.", h=5, legend_rows=1)
    ax.barh(list(y), d["n"], color=C[1], alpha=0.45, height=0.75); ax.barh(list(y), d["mine"], color=C[1], height=0.75)
    ax.set_yticks(list(y)); ax.set_yticklabels([truncate(s) for s in d["thread"]]); ax.invert_yaxis()
    for i, v in enumerate(d["n"]):
        strong(ax, v + d["n"].max() * 0.01, i, fmtN(int(v)), va="center")
    ax.yaxis.grid(False); ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.set_xlim(0, d["n"].max() * 1.1)
    legend_below(fig, [("Sent by this account", C[1]), ("Received", mpl.colors.to_rgba(C[1], 0.45))])
    done(fig, "c-threads")
run(draw_threads, needs=["top_threads"], where="Chapter 3 chart")

def draw_recip():
    d = R["recip_hist"]; y = range(len(d))
    fig, ax = card("Who does the talking", "Direct threads by the share of messages this account sent.", h=5)
    ax.barh(list(y), d["n"], color=[C[1] if lo >= 0.5 else MUTED for lo in d["lo"]], height=0.75)
    ax.set_yticks(list(y)); ax.set_yticklabels([f"{fmtP(lo)}–{fmtP(hi)}" for lo, hi in zip(d["lo"], d["hi"])]); ax.invert_yaxis()
    for i, v in enumerate(d["n"]):
        strong(ax, v + max(1, d["n"].max()) * 0.01, i, fmtN(int(v)), va="center")
    ax.yaxis.grid(False); ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.set_xlim(0, max(1, d["n"].max()) * 1.1)
    ax.set_xlabel("share of messages sent by this account → number of threads", loc="left")
    done(fig, "c-recip")
run(draw_recip, needs=["recip_hist"], where="Chapter 3 chart")

## Chapter 4 — An evening habit

**The economics question.** Attention is a scarce input and it has a timetable: advertisers pay more for some hours
than others, notifications are scheduled to catch you when you are most likely to respond, and the recommender learns
your routine. Messages are the densest time series in the export, so they show your rhythm best: by weekday and hour
(calc `heatmap`), and against the hourly profile of likes, story reactions and saves (`by_hour`, `rhythm`), all in
Rome time.

In [ ]:
def calc_rhythm():
    ev = T["events"]; m = ev[ev["kind"] == "message"]
    if not len(m):
        return skip("Chapter 4", "no messages")
    hm = m.groupby(["weekday_local", "hour_local"]).size()
    R["heatmap"] = pd.DataFrame([dict(weekday=d, hour=h, n=int(hm.get((d, h), 0))) for d in range(7) for h in range(24)])
    rows = []
    for kind in ["message", "like", "story_like", "save"]:
        e = ev[ev["kind"] == kind]
        if len(e):
            byh = e.groupby("hour_local").size().reindex(range(24), fill_value=0)
            rows += [dict(kind=kind, hour=h, share=byh[h] / len(e)) for h in range(24)]
    R["by_hour"] = pd.DataFrame(rows)
    byh = m.groupby("hour_local").size().reindex(range(24), fill_value=0)
    peak = int(byh.idxmax())
    sh = lambda lo, hi: ((m["hour_local"] >= lo) & (m["hour_local"] < hi)).sum() / len(m)
    def wk(k):
        e = ev[ev["kind"] == k]
        return (e["weekday_local"] >= 5).sum() / len(e) if len(e) else float("nan")
    R["rhythm"] = dict(peak_hour=f"{peak:02d}:00–{peak + 1:02d}:00", evening_pct=sh(19, 24), night_pct=sh(0, 6), weekend_msg_pct=wk("message"), weekend_story_pct=wk("story_like"))
    say("Messaging peaks at [rhythm.peak_hour]; [rhythm.evening_pct] of messages are sent between 19:00 and midnight and only [rhythm.night_pct] between midnight and 06:00 (times are Rome time). Weekends take [rhythm.weekend_msg_pct] of messages but [rhythm.weekend_story_pct] of story reactions.")
run(calc_rhythm, needs=["events"], where="Chapter 4")

In [ ]:
def draw_heat():
    d = R["heatmap"]; mx = max(1, d["n"].max())
    fig, ax = card("Messages by weekday and hour", "Darker cells hold more messages.", h=2.6, pad_top=0.2)
    for r in d.itertuples():
        ax.add_patch(Rectangle((r.hour + 0.05, r.weekday + 0.05), 0.9, 0.9, facecolor=C[1], alpha=0.06 + 0.94 * math.sqrt(r.n / mx), edgecolor="none"))
    ax.set_xlim(0, 24); ax.set_ylim(7, 0); ax.yaxis.grid(False); ax.spines["bottom"].set_visible(False)
    ax.set_yticks([i + 0.5 for i in range(7)]); ax.set_yticklabels(WD)
    ax.set_xticks([h + 0.5 for h in [0, 6, 12, 18, 23]]); ax.set_xticklabels([f"{h:02d}" for h in [0, 6, 12, 18, 23]]); ax.xaxis.tick_top()
    done(fig, "c-heat")
run(draw_heat, needs=["heatmap"], where="Chapter 4 chart")

def draw_hour():
    d = R["by_hour"]; kinds = [k for k in ["message", "like", "story_like", "save"] if k in set(d["kind"])]
    fig, ax = card("Hour of day, by type of action", "Each line is the share of that action's total falling in each hour.", h=3.2, legend_rows=1)
    for k in kinds:
        i = ["message", "like", "story_like", "save"].index(k); s = d[d["kind"] == k]
        ax.plot(s["hour"], s["share"], color=C[i], linewidth=2.5 if k == "message" else 1.5, solid_capstyle="round")
    ax.set_xticks([0, 6, 12, 18, 23]); ax.set_xticklabels([f"{h:02d}:00" for h in [0, 6, 12, 18, 23]]); ax.set_xlim(0, 23)
    ax.set_ylim(0, d["share"].max() * 1.08); ax.yaxis.set_major_locator(MaxNLocator(3)); pct_axis(ax); ax.spines["bottom"].set_visible(False)
    legend_below(fig, [(KIND_LABEL[k], C[["message", "like", "story_like", "save"].index(k)]) for k in kinds])
    done(fig, "c-hour")
run(draw_hour, needs=["by_hour"], where="Chapter 4 chart")

## Chapter 5 — The audience froze; the following kept growing

**The economics question.** The social graph is the platform's original asset, and it has two sides: the accounts you
chose to follow (what you want to see) and the accounts that follow you (who listens to you). When the first keeps
growing and the second stops, the account has turned from a node that produces into a node that consumes — and the
graph that the recommender competes with is the following, not the audience. The chapter counts relationships by the
year they started (calc `graph_by_year`) and measures reciprocity: how many accounts are both, and in which direction
the follow-back goes (`graph_tot`).

In [ ]:
def calc_graph():
    g = T["graph"]; years = sorted(g["year"].unique())
    by = g.groupby(["direction", "year"]).size()
    R["graph_by_year"] = pd.DataFrame([dict(direction=dr, year=int(y), n=int(by.get((dr, y), 0))) for dr in ["follower", "following"] for y in years])
    fo, fi = g[g["direction"] == "follower"], g[g["direction"] == "following"]
    mutual = int(flag(fo["mutual"]).sum()); maxy = int(g["year"].max()); miny = int(g["year"].min())
    R["graph_tot"] = dict(followers=len(fo), following=len(fi), mutual=mutual, followback_pct=mutual / len(fo) if len(fo) else float("nan"),
                          followed_back_pct=mutual / len(fi) if len(fi) else float("nan"), first_year=miny,
                          followers_first=int((fo["year"] == miny).sum()), followers_last2=int((fo["year"] >= maxy - 1).sum()))
    say("[graph_tot.followers_first] of the [graph_tot.followers] followers arrived in [graph_tot.first_year], the first year; [graph_tot.followers_last2] arrived in the last two years. Following, meanwhile, grew every year to [graph_tot.following]. Only [graph_tot.mutual] accounts are both: the account follows [graph_tot.followback_pct] of its followers, and [graph_tot.followed_back_pct] of the accounts it follows follow it back. The graph that matters now is the one it consumes from, not the one that listens.")
run(calc_graph, needs=["graph"], where="Chapter 5")

def draw_graph():
    d = R["graph_by_year"]; years = sorted(d["year"].unique()); w = 0.36
    fig, ax = card("Accounts gained per year", "New followers and new accounts followed, by the year the relationship started.", h=3.5, legend_rows=1)
    for i, (dr, col) in enumerate([("follower", C[0]), ("following", C[1])]):
        s = d[d["direction"] == dr].set_index("year")["n"].reindex(years, fill_value=0)
        ax.bar([y + (i - 0.5) * w for y in years], s.values, width=w * 0.92, color=col)
    year_ticks(ax, years); count_axis(ax); ax.spines["bottom"].set_visible(False); ax.set_ylim(0, max(1, d["n"].max()) * 1.05)
    legend_below(fig, [("New followers", C[0]), ("New accounts followed", C[1])])
    done(fig, "c-graph")
run(draw_graph, needs=["graph_by_year"], where="Chapter 5 chart")

## Chapter 6 — What the platform chose to show (the retained feed window)

**The economics question.** In its first decade Instagram showed you what the accounts you follow posted; since about
2020 a growing share of the feed, and almost all of Reels, is picked by a recommender from accounts you do *not*
follow. The export keeps only a few days of viewing history, but that window is the only direct view of the *supply*
side of your attention: items shown per day, by source (calc `feed_by_day`), the composition of videos and posts
(`feed_comp`, `feed_tot`), and the accounts most present in what you were shown (`top_accounts`).

In [ ]:
def calc_feed():
    f = T["feed"]; days = sorted(f["day"].unique())
    by = f.groupby(["day", "source"]).size()
    R["feed_by_day"] = pd.DataFrame([dict(day=dd, source=s, n=int(by.get((dd, s), 0))) for dd in days for s in ["followed", "not followed", "ad"]])
    bk = f.groupby(["kind", "source"]).size()
    R["feed_comp"] = pd.DataFrame([dict(kind=k, source=s, n=int(bk.get((k, s), 0))) for k in ["video", "post"] for s in ["followed", "not followed"]])
    v, p, a = (f[f["kind"] == k] for k in ["video", "post", "ad"])
    nf = lambda x: (x["source"] == "not followed").sum() / len(x) if len(x) else float("nan")
    R["feed_tot"] = dict(videos=len(v), posts=len(p), ads=len(a), organic_per_ad=round((len(v) + len(p)) / len(a), 1) if len(a) else float("nan"),
                         video_not_followed_pct=nf(v), post_not_followed_pct=nf(p), n_days=len(days), first_day=days[0], last_day=days[-1],
                         distinct_public=v.loc[v["source"] == "not followed", "account"].nunique(), distinct_network=v.loc[v["source"] == "followed", "account"].nunique())
    org = f[(f["kind"] != "ad") & f["account"].notna()]
    cnt = org.groupby("account", sort=False).agg(n=("kind", "size"), source=("source", "first")).reset_index()
    R["top_accounts"] = cnt.sort_values("n", ascending=False, kind="stable").head(12).reset_index(drop=True)
    print(f"Feed window: {days[0]} to {days[-1]} ({len(days)} days with items).")
    say("The export keeps only a few days of viewing history: [feed_tot.videos] videos, [feed_tot.posts] posts and [feed_tot.ads] ads in [feed_tot.n_days] days, one ad for every [feed_tot.organic_per_ad] organic items, a ratio that barely moves from day to day. [feed_tot.video_not_followed_pct] of the videos came from accounts this user does not follow \u2014 [feed_tot.distinct_public] distinct public accounts against [feed_tot.distinct_network] from the network \u2014 while for posts the share is [feed_tot.post_not_followed_pct]. Video is where recommendation replaces the social graph.")
run(calc_feed, needs=["feed"], where="Chapter 6")

In [ ]:
def draw_feed_day():
    d = R["feed_by_day"]; days = sorted(d["day"].unique()); x = range(len(days))
    fig, ax = card("Items shown per day", "Videos and posts by whether the author is followed, and ads.", h=3.5, legend_rows=1)
    bottom = pd.Series(0.0, index=days)
    for s, col in [("followed", C[0]), ("not followed", C[1]), ("ad", C[3])]:
        v = d[d["source"] == s].set_index("day")["n"].reindex(days, fill_value=0)
        ax.bar(list(x), v.values, bottom=bottom.values, color=col, width=0.7); bottom = bottom + v
    ax.set_xticks(list(x)); ax.set_xticklabels([f"{pd.Timestamp(dd).day} {pd.Timestamp(dd):%b}" for dd in days])
    count_axis(ax); ax.spines["bottom"].set_visible(False); ax.set_ylim(0, max(1, bottom.max()) * 1.05)
    legend_below(fig, [("From followed accounts", C[0]), ("From accounts not followed", C[1]), ("Ads", C[3])])
    done(fig, "c-feedday")
run(draw_feed_day, needs=["feed_by_day"], where="Chapter 6 chart")

def draw_feed_comp():
    d = R["feed_comp"]
    fig, ax = card("Where the feed comes from", "Composition of videos and posts shown.", h=1.6, legend_rows=1)
    for i, k in enumerate(["video", "post"]):
        f = int(d[(d["kind"] == k) & (d["source"] == "followed")]["n"].sum()); nf = int(d[(d["kind"] == k) & (d["source"] == "not followed")]["n"].sum()); tot = (f + nf) or 1
        ax.barh(i, f / tot, color=C[0], height=0.65); ax.barh(i, nf / tot, left=f / tot, color=C[1], height=0.65)
        for x0, x1, n in [(0, f / tot, f), (f / tot, 1, nf)]:
            if x1 - x0 > 0.12:
                ax.text((x0 + x1) / 2, i, fmtP(n / tot), color="white", ha="center", va="center", fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Videos", "Posts"], color=INK); ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xticks([])
    ax.yaxis.grid(False); ax.spines["bottom"].set_visible(False)
    legend_below(fig, [("From followed accounts", C[0]), ("From accounts not followed (recommended)", C[1])])
    done(fig, "c-feedcomp")
run(draw_feed_comp, needs=["feed_comp"], where="Chapter 6 chart")

def hbar(items, key, val, colors, title, note, chart_id, legend=None, h=5):
    """The page's hbar: horizontal bars, label left (truncated at 20 characters), count right."""
    fig, ax = card(title, note, h=h, legend_rows=1 if legend else 0)
    y = list(range(len(items)))
    ax.barh(y, items[val], color=colors if isinstance(colors, str) else list(colors), height=0.75)
    ax.set_yticks(y); ax.set_yticklabels([truncate(s) for s in items[key]]); ax.invert_yaxis()
    mx = max(1, items[val].max())
    for i, v in enumerate(items[val]):
        strong(ax, v + mx * 0.01, i, fmtN(int(v)), va="center")
    ax.yaxis.grid(False); ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.set_xlim(0, mx * 1.1)
    if legend:
        legend_below(fig, legend)
    done(fig, chart_id)

run(lambda: hbar(R["top_accounts"], "account", "n", [C[0] if s == "followed" else C[1] for s in R["top_accounts"]["source"]], "Accounts most present in the feed", "Videos and posts shown per account; public accounts the user does not follow are named, followed accounts are pseudonymised.", "c-topacc",
                 legend=[("Followed (pseudonymised)", C[0]), ("Not followed (public account)", C[1])]), needs=["top_accounts"], where="Chapter 6 chart")

## Chapter 7 — The advertisers, and who holds the data

**The economics question.** Instagram's revenue is advertising, and the export shows two layers of it. The ads actually
shown in the window tell you who bought your attention (calc `top_ads`, `ads_tot`). The list of advertisers that
uploaded a customer list containing you — your e-mail or phone from a purchase, a newsletter, a loyalty card, or a
visit tracked by the Meta pixel — tells you who already held data on you *before* the ad was placed: targeting from
off-platform data, in a market you were never a party to (`advertisers`).

In [ ]:
def calc_ads():
    f = T["feed"]; a = f[(f["kind"] == "ad") & f["account"].notna()]
    cnt = a.groupby("account", sort=False).size().sort_values(ascending=False, kind="stable")
    R["top_ads"] = pd.DataFrame({"account": cnt.index[:15], "n": cnt.values[:15].astype(int)})
    R["ads_tot"] = dict(n_advertisers=len(cnt), top_n=int(cnt.max()) if len(cnt) else 0,
                        n_holders=T["advertisers"]["advertiser"].nunique() if T.get("advertisers") is not None else None)
    say("The [feed_tot.ads] ads came from [ads_tot.n_advertisers] advertisers; the most frequent one alone placed [ads_tot.top_n]. Separately, [ads_tot.n_holders] advertisers uploaded customer lists to Meta that contained this user: brands next to media-buying agencies \u2014 the agencies that run campaigns for many brands hold the data too.")
run(calc_ads, needs=["feed"], where="Chapter 7")

run(lambda: hbar(R["top_ads"], "account", "n", C[3], "Ads shown, by advertiser", "The fifteen advertiser accounts with most ads in the window.", "c-ads"), needs=["top_ads"], where="Chapter 7 chart")

def table_advertisers():
    how = {"uploaded_your_contact_details": "uploaded a customer list", "tracked_you_on_their_site_or_app": "tracked a visit to their site or app", "store_visited": "in-store visit"}
    a = T["advertisers"].copy()
    a["how"] = a["how"].map(lambda h: how.get(h, h)) if "how" in a else ""
    print("Advertisers that uploaded a list matching this user \u2014 From Meta's own disclosure in the export.")
    display(a.sort_values("advertiser", key=lambda s: s.astype(str).str.lower()).reset_index(drop=True).rename(columns={"advertiser": "Advertiser", "how": "How they got the match"}))
run(table_advertisers, needs=["advertisers"], where="Chapter 7 table")

## Chapter 8 — Liking is loyalty, saving is discovery

**The economics question.** Not all engagement is the same signal. A like is cheap and social — it goes to the same few
friends, over and over; a save is rarer and private — it marks something you want to come back to, and it is where the
recommender's suggestions actually land. The chapter draws concentration curves: accounts ranked by the actions they
received from you, against the cumulative share of those actions (calc `lorenz`, `lorenz_sum`), and the share of
likes and saves that went to public accounts you do not follow (`outside`).

In [ ]:
def calc_lorenz():
    ev = T["events"]; rows = []; summ = {}
    for kind in ["like", "story_like", "save"]:
        e = ev[(ev["kind"] == kind) & ev["actor"].notna() & (ev["actor"].astype(str) != "")]
        counts = sorted(e["actor"].value_counts().values, reverse=True); tot = sum(counts)
        rows.append(dict(kind=kind, rank=0, x=0.0, y=0.0)); cum = 0
        for i, c in enumerate(counts):
            cum += c; rows.append(dict(kind=kind, rank=i + 1, x=(i + 1) / len(counts), y=cum / tot))
        summ[kind] = dict(share=sum(counts[:5]) / tot if tot else float("nan"), n=tot, accounts=len(counts))
    R["lorenz"] = pd.DataFrame(rows)
    R["lorenz_sum"] = dict(likes_top5=summ["like"]["share"], likes_n=summ["like"]["n"], story_top5=summ["story_like"]["share"], saves_top5=summ["save"]["share"], saves_accounts=summ["save"]["accounts"])
    sh = lambda k: flag(ev.loc[ev["kind"] == k, "actor_public"]).mean() if (ev["kind"] == k).any() else float("nan")
    R["outside"] = dict(like_public_pct=sh("like"), save_public_pct=sh("save"))
    say("Likes go to a few accounts: [lorenz_sum.likes_top5] of the [lorenz_sum.likes_n] post likes went to five accounts, and [lorenz_sum.story_top5] of story reactions to five accounts. Saves are spread almost evenly over [lorenz_sum.saves_accounts] accounts, the top five taking [lorenz_sum.saves_top5], and [outside.save_public_pct] of them come from accounts this user does not follow, against [outside.like_public_pct] of likes. The like is a message to a friend; the save is a note to self.")
run(calc_lorenz, needs=["events"], where="Chapter 8")

def draw_lorenz():
    d = R["lorenz"]
    fig, ax = card("Concentration of attention", "Accounts ranked by how many actions they received; the curve shows the cumulative share. A straight diagonal would be perfectly even.", h=4.2, legend_rows=1)
    ax.plot([0, 1], [0, 1], color=REF, linestyle=(0, (4, 4)), linewidth=1)
    for i, k in enumerate(["like", "story_like", "save"]):
        p = d[d["kind"] == k].sort_values("x")
        if len(p) < 2:
            continue
        ax.plot(p["x"], p["y"], color=C[i], linewidth=2)
        at = p.iloc[(p["x"] - 0.2).abs().argmin()]
        ax.plot([at["x"]], [at["y"]], "o", color=C[i], markersize=5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); pct_axis(ax, "x", [0, .25, .5, .75, 1]); pct_axis(ax, "y", [0, .5, 1]); ax.set_yticks([0, .25, .5, .75, 1], minor=True)
    ax.yaxis.grid(True, which="both", color=GRID); ax.spines["bottom"].set_visible(False)
    ax.set_xlabel("share of accounts (most-liked first)")
    legend_below(fig, [(KIND_LABEL[k], C[i]) for i, k in enumerate(["like", "story_like", "save"])])
    done(fig, "c-lorenz")
run(draw_lorenz, needs=["lorenz"], where="Chapter 8 chart")

## Chapter 9 — The other side of the counter

**The economics question.** Every account is also, in a small way, a creator: it publishes, and the platform measures
what came back. The creator dashboard shows likes and comments per post; put next to the millions-strong audiences of
the accounts the feed is assembled from, it is the clearest picture of the long tail — most supply on the platform
reaches tens of people, and the platform's business is in the head of the distribution. The chapter plots likes
received per own post, in order of publication (calc `post_insights`, `ins_tot`).

In [ ]:
def calc_ins():
    pi = T["post_insights"]; likes = sorted(pi["likes"].dropna().astype(int))
    R["ins_tot"] = dict(n_posts=len(pi), median_likes=likes[len(likes) // 2] if likes else None, max_likes=likes[-1] if likes else None,
                        total_comments=int(pi["comments"].fillna(0).sum()) if "comments" in pi else 0)
    say("For its own [ins_tot.n_posts] posts with insights, the account received a median of [ins_tot.median_likes] likes per post, a maximum of [ins_tot.max_likes], and [ins_tot.total_comments] comments in total. With [totals.followers] followers, this is what the creator dashboard shows a small account: numbers in the tens, while the feed it consumes is assembled from accounts with audiences in the millions.")
run(calc_ins, needs=["post_insights"], where="Chapter 9")

def draw_ins():
    pi = T["post_insights"]; likes = pi["likes"].fillna(0).astype(int).values; x = range(len(pi))
    fig, ax = card("Likes received per post", "One bar per published post, in order of publication.", h=3.2)
    ax.bar(list(x), likes, color=C[0], width=0.8)
    yrs = pi["timestamp"].astype(str).str[:4]; first = ~yrs.duplicated()
    ax.set_xticks([i for i in x if first.iloc[i]]); ax.set_xticklabels(list(yrs[first]), ha="left")
    count_axis(ax, nticks=3); ax.spines["bottom"].set_visible(False); ax.set_xlim(-0.6, len(pi) - 0.4); ax.set_ylim(0, max(1, likes.max()) * 1.05)
    done(fig, "c-ins")
run(draw_ins, needs=["post_insights"], where="Chapter 9 chart")

## Chapter 10 — The ad as a supply chain: who pays, who posts, who discloses

**The economics question.** An ad on Instagram is a small supply chain: a brand pays, an account posts (its own, or a
creator's on the brand's behalf), the platform places, and the law asks for a disclosure — in Italy, `#adv`. The
chapter follows the chain for the ads shown in the window: who posts for whom (calc `ads_by_adv`), which hashtags the
ads carry and whether they disclose (`ads_tags`), how long ad captions are compared with the organic items shown next
to them (`ads_caplen`), and whether the advertisers also appear among the accounts you consume or among the businesses
holding your data (`ads_sum`).

In [ ]:
ADS_BINS = [(0, 10, "0–10"), (11, 25, "11–25"), (26, 50, "26–50"), (51, 100, "51–100"), (101, 200, "101–200"), (201, float("inf"), "200+")]
ADS_DISC = ["adv", "ad", "ads", "sponsored", "sponsorizzato", "pubblicita", "pubblicità", "gifted", "paidpartnership"]

def calc_adschain():
    ads = T["ads"]; a = ads[ads["kind"] == "ad"]; org = ads[ads["kind"] != "ad"]; n = len(a)
    same = flag(a["same_account"]); disc = flag(a["has_disclosure"])
    creator, own = a[~same], a[same]
    by_adv = a.groupby("advertiser", sort=False).size().sort_values(ascending=False, kind="stable")
    top = by_adv.index[0]; top_row = a[a["advertiser"] == top].iloc[0]
    med = lambda s: float(pd.Series(s).median()) if len(s) else 0
    inl = a["in_customer_lists"].notna() & (a["in_customer_lists"].astype(str) != "")
    nv = int(flag(a["in_videos_watched"]).sum())
    holders = T["advertisers"]["advertiser"].nunique() if T.get("advertisers") is not None else 0
    R["ads_sum"] = dict(n_ads=n, n_advertisers=len(by_adv), n_owners=a["owner"].nunique(), n_creator=len(creator), creator_pct=len(creator) / n, n_own=len(own),
                        n_brand_partners_using_creators=creator["brand_partner"].nunique(), n_disclosed=int(disc.sum()), disclosed_pct=disc.sum() / n,
                        n_creator_disclosed=int(disc[~same].sum()), creator_disclosed_pct=disc[~same].mean() if len(creator) else 0,
                        n_own_disclosed=int(disc[same].sum()), n_own_disclosed_accounts=own[disc[same]]["owner"].nunique(),
                        top_advertiser=top_row["brand_partner_name"] or top_row["owner_name"] or top, top_advertiser_n=int(by_adv.iloc[0]),
                        median_words_ad=med(a["n_words_caption"]), median_words_organic=med(org["n_words_caption"]), n_organic_items=len(org),
                        n_organic_disclosed=int(flag(org["has_disclosure"]).sum()), n_appears_organic=int(flag(a["appears_organic"]).sum()),
                        n_advertisers_organic=a[flag(a["appears_organic"])]["advertiser"].nunique(), n_in_lists=int(inl.sum()), n_advertisers_in_lists=a[inl]["advertiser"].nunique(),
                        n_list_holders=holders, n_in_videos_watched=nv, in_videos_watched_pct=nv / n)
    # who posts, for whom (top 12 advertisers)
    kind = pd.Series("own", index=a.index).mask(~same, "creator").mask(same & disc, "own_disclosed")
    tab = pd.crosstab(a["advertiser"], kind).reindex(columns=["own", "own_disclosed", "creator"], fill_value=0).reindex(by_adv.index)
    tab["n"] = tab.sum(axis=1); tab["name"] = [a[a["advertiser"] == adv].iloc[0]["brand_partner_name"] or a[a["advertiser"] == adv].iloc[0]["owner_name"] or adv for adv in tab.index]
    R["ads_by_adv"] = tab.reset_index().head(12)
    # hashtags (one count per ad)
    tags = a["hashtags"].fillna("").astype(str).apply(lambda s: sorted(set(s.split()))).explode().dropna()
    tc = tags.value_counts()
    tg = pd.DataFrame({"tag": ["#" + t for t in tc.index], "n": tc.values.astype(int), "is_disclosure": [t in ADS_DISC for t in tc.index]})
    tg["_t"] = tc.index
    R["ads_tags"] = tg.sort_values(["n", "_t"], ascending=[False, True], kind="stable").drop(columns="_t").head(12).reset_index(drop=True)
    # caption length bins
    rows = []
    for k, it in [("ad", a), ("organic", org)]:
        for lo, hi, label in ADS_BINS:
            c = int(((it["n_words_caption"] >= lo) & (it["n_words_caption"] <= hi)).sum()); rows.append(dict(kind=k, bin=label, n=c, share=c / len(it) if len(it) else 0))
    R["ads_caplen"] = pd.DataFrame(rows)
    print(f"Most frequent disclosure hashtag among the ads: {R['ads_tags'][R['ads_tags']['is_disclosure']]['tag'].iloc[0] if R['ads_tags']['is_disclosure'].any() else 'none'}.")
    say("The [ads_sum.n_ads] ads shown in [feed_tot.n_days] days were paid for by [ads_sum.n_advertisers] advertisers but posted by [ads_sum.n_owners] accounts: [ads_sum.n_creator] ads ([ads_sum.creator_pct]) were posted by a creator on behalf of a brand partner, and [ads_sum.n_own_disclosed] more came from an account that names its sponsor only through #adv, with no brand partner recorded. In all, [ads_sum.n_disclosed] ads ([ads_sum.disclosed_pct]) carry a sponsorship hashtag, most often #adv, the Italian disclosure; ad captions are short, a median of [ads_sum.median_words_ad] words against [ads_sum.median_words_organic] for the organic items shown. The chains do not touch: [ads_sum.n_advertisers_organic] of the advertisers or their creators appear as authors of organic content, [ads_sum.n_advertisers_in_lists] of them are among the [ads_sum.n_list_holders] advertisers that uploaded a customer list containing this user \u2014 yet [ads_sum.n_in_videos_watched] of the [ads_sum.n_ads] ads are also logged in the export as videos the user watched.")
run(calc_adschain, needs=["ads"], where="Chapter 10")

In [ ]:
def draw_ads_adv():
    d = R["ads_by_adv"]; y = list(range(len(d))); left = pd.Series(0.0, index=d.index)
    keys = [("own", C[3], "Posted by the brand's own account"), ("own_disclosed", C[2], "Own account, sponsor named only by #adv"), ("creator", C[1], "Posted by a creator for a brand partner")]
    fig, ax = card("Who posts the ad, for whom", "The twelve advertisers with most ads (the brand partner, or the posting account when none is recorded), split by who posted the content.", h=5, legend_rows=2)
    for k, col, _l in keys:
        ax.barh(y, d[k], left=left, color=col, height=0.75); left = left + d[k]
    ax.set_yticks(y); ax.set_yticklabels([truncate(s) for s in d["advertiser"]]); ax.invert_yaxis()
    for i, v in enumerate(d["n"]):
        strong(ax, v + d["n"].max() * 0.01, i, fmtN(int(v)), va="center")
    ax.yaxis.grid(False); ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.set_xlim(0, d["n"].max() * 1.1)
    legend_below(fig, [(l, c) for _k, c, l in keys], ncol=2)
    done(fig, "c-adsadv")
run(draw_ads_adv, needs=["ads_by_adv"], where="Chapter 10 chart")

run(lambda: hbar(R["ads_tags"], "tag", "n", [C[3] if d else MUTED for d in R["ads_tags"]["is_disclosure"]], "Hashtags in the ads", "Ads carrying each hashtag, from the hashtag field and the caption. #adv is the Italian sponsored-content disclosure.", "c-adstags",
                 legend=[("Sponsorship disclosure", C[3]), ("Other hashtag", MUTED)]), needs=["ads_tags"], where="Chapter 10 chart")

def draw_ads_cap():
    d = R["ads_caplen"]; bins = [b[2] for b in ADS_BINS]; x = range(len(bins)); w = 0.36
    fig, ax = card("Caption length: ads versus organic items", "Share of ads, and of organic videos and posts shown, by words in the caption; the copies of ads that the export also lists among watched videos are removed from the organic side.", h=3.2, legend_rows=1)
    for i, (k, col) in enumerate([("ad", C[3]), ("organic", C[0])]):
        s = d[d["kind"] == k].set_index("bin")["share"].reindex(bins, fill_value=0)
        ax.bar([j + (i - 0.5) * w for j in x], s.values, width=w * 0.92, color=col)
    ax.set_xticks(list(x)); ax.set_xticklabels(bins); ax.set_xlabel("words in the caption → share of items")
    ax.set_ylim(0, max(0.01, d["share"].max()) * 1.08); ax.yaxis.set_major_locator(MaxNLocator(4)); pct_axis(ax); ax.spines["bottom"].set_visible(False)
    legend_below(fig, [("Ads", C[3]), ("Organic videos and posts", C[0])])
    done(fig, "c-adscap")
run(draw_ads_cap, needs=["ads_caplen"], where="Chapter 10 chart")

## Chapter 11 — Conversations do not die; they slow down

**The economics question.** Network effects are usually measured by how many people are on a platform; what keeps a
user is how many of *their* people are still active there, and how the exchange with them feels. The chapter follows
each conversation through the years — alive, born, ended (calc `convo_life`) — then measures its texture: how long a
reply takes on each side (`convo_texture`), and what a burst of messages looks like — a lone shared reel with no answer,
a lone text, or an actual exchange (`convo_sessions`). A thread that is still alive but answered days later, with a
shared post instead of a sentence, is the platform's content travelling along a friendship rather than the friendship
itself.

In [ ]:
CV_CAP = 7 * 24 * 60   # reply latency is capped at 7 days (minutes)

def lat_label(mins):
    if mins is None or (isinstance(mins, float) and math.isnan(mins)):
        return "—"
    unit = lambda v, u: f"{v:g} {u}{'' if v == 1 else 's'}"
    rnd = lambda x: math.floor(x + 0.5)
    if mins >= CV_CAP: return "more than 7 days"
    if mins >= 1440: return unit(rnd(mins / 144) / 10, "day")
    if mins >= 60: return unit(rnd(mins / 6) / 10, "hour")
    return unit(rnd(mins), "minute")

def calc_convo():
    cv = T["convo"]; years = sorted(cv["year"].unique()); last = years[-1]
    act = {y: set(cv.loc[cv["year"] == y, "thread"]) for y in years}
    first = cv.groupby("thread")["thread_first_year"].first(); lasty = cv.groupby("thread")["thread_last_year"].first()
    rows = []
    for y in years:
        a = act[y]; prev = act.get(y - 1, set())
        retained = len(a & prev); new = int(sum(first[t] == y for t in a)); ended = int(sum(lasty[t] == y for t in a))
        rows.append(dict(year=y, active=len(a), new=new, retained=retained, prev_active=len(prev), retention_pct=retained / len(prev) if prev else 0,
                         ended=0 if y == last else ended, open=ended if y == last else 0, is_last=y == last))
    life = pd.DataFrame(rows); R["convo_life"] = life
    peak = life.sort_values("active", ascending=False, kind="stable").iloc[0]; tail = life.tail(4)
    R["convo_life_tot"] = dict(n_threads=cv["thread"].nunique(), peak_year=int(peak["year"]), peak_active=int(peak["active"]), new_peak=int(peak["new"]),
                               new_last3=int(life.tail(3)["new"].sum()), retention_last4_pct=tail["retained"].sum() / tail["prev_active"].sum() if tail["prev_active"].sum() else float("nan"),
                               first_year=int(life.iloc[0]["year"]), last_year=int(last), open_last=int(life.iloc[-1]["open"]), active_last=int(life.iloc[-1]["active"]))
    def texture(m):
        rep = m[m["reply_latency_min"].notna()]; me = rep[flag(rep["is_me"])]["reply_latency_min"]; ot = rep[~flag(rep["is_me"])]["reply_latency_min"]
        lm, lo = (me.median() if len(me) else float("nan")), (ot.median() if len(ot) else float("nan"))
        return dict(n=len(m), lat_me=lm, lat_other=lo, lat_me_label=lat_label(lm), lat_other_label=lat_label(lo), n_rep_me=len(me), n_rep_other=len(ot),
                    me_capped_pct=(me >= CV_CAP).mean() if len(me) else 0, share_pct=flag(m["is_share"]).mean(), question_pct=flag(m["has_question"]).mean(),
                    group_pct=(m["thread_type"] == "group").mean(), direct_pct=(m["thread_type"] == "direct").mean())
    R["convo_texture"] = pd.DataFrame([dict(year=y, **texture(cv[cv["year"] == y])) for y in years])
    R["convo_texture_tot"] = texture(cv)
    firsts = cv[cv["session_pos"] == 1]
    def sessions(s):
        sizes = s["session_size"]; single = s[sizes == 1]; ss = int(flag(single["is_share"]).sum()); n = len(s)
        return dict(sessions=n, messages=int(sizes.sum()), mean_size=round(sizes.mean(), 2), median_size=float(sizes.median()), max_size=int(sizes.max()), single=len(single),
                    single_pct=len(single) / n, single_share=ss, single_share_pct=ss / n, single_other=len(single) - ss, single_other_pct=(len(single) - ss) / n,
                    multi=n - len(single), multi_pct=(n - len(single)) / n)
    R["convo_sessions"] = pd.DataFrame([dict(year=y, **sessions(firsts[firsts["year"] == y])) for y in years])
    st = sessions(firsts); single = firsts[firsts["session_size"] == 1]
    st["single_by_me_pct"] = flag(single["is_me"]).mean() if len(single) else 0
    R["convo_sessions_tot"] = st
    say("Of [convo_life_tot.n_threads] conversations with at least one message, [convo_life_tot.new_peak] began in [convo_life_tot.peak_year], the year with most threads alive ([convo_life_tot.peak_active]); only [convo_life_tot.new_last3] began in the last three years. Since then the cast has been stable: [convo_life_tot.retention_last4_pct] of the threads active in one year are still active the next, and [convo_life.active@-1] were open in [convo_life.year@-1]. What changed is the pace. The other side answers after a median [convo_texture_tot.lat_other_label]; this account takes [convo_texture_tot.lat_me_label], up from [convo_texture.lat_me_label@2] in [convo_texture.year@2] to [convo_texture.lat_me_label@-1] in [convo_texture.year@-1]. [convo_sessions_tot.single_pct] of the [convo_sessions_tot.sessions] bursts of messages are a single message with no reply within half an hour, and a lone shared post or reel grew from [convo_sessions.single_share_pct@2] of bursts in [convo_sessions.year@2] to [convo_sessions.single_share_pct@-1] in [convo_sessions.year@-1]. Only [convo_texture_tot.question_pct] of messages contain a question and [convo_texture_tot.group_pct] sit in a group: the same few people, exchanging the platform's content at days of distance.")
run(calc_convo, needs=["convo"], where="Chapter 11")

In [ ]:
def draw_convo_life():
    d = R["convo_life"]
    fig, ax = card("Threads alive, born and ended, per year", "A thread is active in a year if it has at least one message that year (Rome time). Above the axis: active threads, split into those already active the year before and those starting that year. Below the axis: threads whose last message falls in that year; the final year of the export is not counted as an ending.", h=3.8, legend_rows=2)
    ax.bar(d["year"], d["retained"], color=C[1], width=0.75); ax.bar(d["year"], d["active"] - d["retained"], bottom=d["retained"], color=C[0], width=0.75)   # as on the page: the top segment runs from retained to active
    ax.bar(d["year"], -d["ended"], color=MUTED, width=0.75)
    for r in d.itertuples():
        strong(ax, r.year, r.active + d["active"].max() * 0.015, fmtN(int(r.active)), ha="center", va="bottom")
    ax.axhline(0, color=AXIS, linewidth=0.8); ax.spines["bottom"].set_visible(False)
    year_ticks(ax, d["year"]); ax.yaxis.set_major_locator(MaxNLocator(5)); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _p: fmtN(int(abs(v)))))
    ax.set_ylim(-max(1, d["ended"].max()) * 1.15, d["active"].max() * 1.15)
    box = dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5)
    ax.text(1, d["active"].max() * 0.03, "active threads", transform=ax.get_yaxis_transform(), ha="right", va="bottom", color=TICK, fontsize=8.5, bbox=box)
    ax.text(1, -d["active"].max() * 0.03, "threads that ended", transform=ax.get_yaxis_transform(), ha="right", va="top", color=TICK, fontsize=8.5, bbox=box)
    legend_below(fig, [("Continued from the year before", C[1]), ("Started this year", C[0]), ("Received its last message this year (below the axis)", MUTED)])
    done(fig, "c-convo-life")
run(draw_convo_life, needs=["convo_life"], where="Chapter 11 chart")

def draw_convo_lat():
    d = R["convo_texture"].dropna(subset=["lat_me", "lat_other"])
    fig, ax = card("How long a reply takes", "Median minutes from a message to the next one by a different sender in the same thread, capped at seven days; log scale. One line for this account's replies, one for everyone else's.", h=3.5, legend_rows=1)
    for k, col in [("lat_me", C[1]), ("lat_other", C[0])]:
        ax.plot(d["year"], d[k].clip(lower=1), color=col, linewidth=2, marker="o", markersize=5)
    ax.set_yscale("log"); ax.set_ylim(1, CV_CAP); ax.set_yticks([1, 10, 60, 1440, CV_CAP]); ax.set_yticklabels(["1 min", "10 min", "1 h", "1 day", "7 days"])
    ax.yaxis.set_minor_locator(NullLocator()); ax.spines["bottom"].set_visible(False); year_ticks(ax, d["year"])
    legend_below(fig, [("Before this account answers", C[1]), ("Before the other side answers", C[0])])
    done(fig, "c-convo-lat")
run(draw_convo_lat, needs=["convo_texture"], where="Chapter 11 chart")

def draw_convo_sess():
    d = R["convo_sessions"]
    fig, ax = card("What a burst of messages looks like", "A burst (session) is a run of messages in one thread less than 30 minutes apart. Share of bursts per year that are a lone share, a lone text, or an actual exchange.", h=3.5, legend_rows=1)
    bottom = pd.Series(0.0, index=d.index)
    for k, col in [("single_share_pct", C[1]), ("single_other_pct", C[3]), ("multi_pct", MUTED)]:
        ax.bar(d["year"], d[k], bottom=bottom, color=col, width=0.75); bottom = bottom + d[k]
    ax.set_ylim(0, 1); pct_axis(ax, ticks=[0, .5, 1]); ax.set_yticks([0, .25, .5, .75, 1], minor=True); ax.yaxis.grid(True, which="both", color=GRID)
    ax.spines["bottom"].set_visible(False); year_ticks(ax, d["year"])
    legend_below(fig, [("A lone share, no reply", C[1]), ("A lone text or other message", C[3]), ("An exchange of two or more messages", MUTED)])
    done(fig, "c-convo-sess")
run(draw_convo_sess, needs=["convo_sessions"], where="Chapter 11 chart")

## Chapter 12 — The feed drifted out of the network, the output dried up

**The economics question.** Two drifts, in opposite directions. On the consumption side, the share of what you react
to that comes from accounts you do not follow is the recommender's share of your taste (calc `taste_rec`,
`taste_rec_sum`) — and saves, the strongest signal of intent, are where it is largest. On the production side, what
the account published shrank and changed form — fewer posts, shorter captions, fewer hashtags — while what it received
in return did not depend on any of it (`taste_output`, `taste_rel`). Finally, taste itself: how concentrated your
likes are on a few accounts each year, and how many of them return from the year before (`taste_conc`,
`taste_conc_sum`). The period cut-offs are set at the top of the cell.

In [ ]:
TASTE_EARLY_MAX, TASTE_LATE_MIN = 2022, 2024     # "over 2018–2022 as a whole ... over 2024–2026"

def calc_taste():
    ta = T["taste"]; is_rec = ~flag(ta["in_network"])
    likes, saves = ta[ta["kind"] == "like"], ta[ta["kind"] == "save"].sort_values("timestamp", kind="stable")
    years = sorted(set(likes["year"]) | set(saves["year"])); seen = set(); rows = []
    for y in years:
        l, s = likes[likes["year"] == y], saves[saves["year"] == y]
        lr, sr = int(is_rec[l.index].sum()), int(is_rec[s.index].sum()); new_acc = new_rec = 0
        for idx, auth in zip(s.index, s["author"]):
            if auth not in seen:
                seen.add(auth); new_acc += 1; new_rec += int(is_rec[idx])
        rows.append(dict(year=y, n_like=len(l), rec_like=lr, share_like=lr / len(l) if len(l) else 0, n_save=len(s), rec_save=sr, share_save=sr / len(s) if len(s) else 0,
                         new_saved_accounts=new_acc, new_saved_rec=new_rec))
    rec = pd.DataFrame(rows); R["taste_rec"] = rec
    l = rec[rec["n_like"] > 0]; s = rec[rec["n_save"] > 0]
    if len(l):
        first, last = l.iloc[0], l.iloc[-1]; peak = l.sort_values("share_like", ascending=False, kind="stable").iloc[0]
        early, late = l[l["year"] <= TASTE_EARLY_MAX], l[l["year"] >= TASTE_LATE_MIN]
        pct = lambda d: d["rec_like"].sum() / d["n_like"].sum() if d["n_like"].sum() else float("nan")
        R["taste_rec_sum"] = dict(first_year=int(first["year"]), first_pct=first["share_like"], last_year=int(last["year"]), last_pct=last["share_like"], peak_year=int(peak["year"]), peak_pct=peak["share_like"],
                                  early_pct=pct(early), late_pct=pct(late), late_n=int(late["n_like"].sum()), early_label=f"{int(l['year'].min())}–{TASTE_EARLY_MAX}", late_label=f"{TASTE_LATE_MIN}–{int(l['year'].max())}",
                                  pre_follow_likes=int(flag(likes["pre_follow"]).sum()))
        if len(s):
            low = s.sort_values("share_save", kind="stable").iloc[0]
            R["taste_rec_sum"].update(save_n=int(s["n_save"].sum()), save_pct=s["rec_save"].sum() / s["n_save"].sum(), save_first_year=int(s.iloc[0]["year"]), save_first_new=int(s.iloc[0]["new_saved_accounts"]),
                                      save_last_year=int(s.iloc[-1]["year"]), save_last_new=int(s.iloc[-1]["new_saved_accounts"]), save_low_year=int(low["year"]), save_low_pct=low["share_save"])
    # own output per year, posts with insights, and the relation between caption and likes received
    posts = ta[ta["kind"] == "post"]
    if len(posts):
        r2 = lambda x: round(float(x), 2) if pd.notna(x) else 0.0
        out = []
        for y in sorted(posts["year"].unique()):
            p = posts[posts["year"] == y]; ins = p[p["likes_received"].notna()]
            out.append(dict(year=int(y), posts=len(p), words=r2(p["n_words_caption"].mean()), hashtags=r2(p["n_hashtags"].mean()), mentions=r2(p["n_mentions"].astype(float).mean()),
                            media=r2(p["n_media"].astype(float).mean()), n_insights=len(ins), likes_mean=r2(ins["likes_received"].astype(float).mean()) if len(ins) else 0.0))
        out = pd.DataFrame(out); R["taste_output"] = out
        pp = posts[posts["likes_received"].notna()]
        R["taste_posts"] = pd.DataFrame(dict(timestamp=pp["timestamp"].values, year=pp["year"].values, words=pp["n_words_caption"].astype(int).values, hashtags=pp["n_hashtags"].astype(int).values,
                                             mentions=pp["n_mentions"].astype(float).values, likes=pp["likes_received"].astype(int).values, comments=pp["comments_received"].astype(float).values))
        tp = R["taste_posts"]
        if len(tp):
            corr = lambda x: round(float(tp[x].astype(float).corr(tp["likes"].astype(float))), 2) if tp[x].std() > 0 and tp["likes"].std() > 0 else 0
            withH, noH = tp[tp["hashtags"] > 0], tp[tp["hashtags"] == 0]; mean = lambda d: round(d["likes"].mean(), 1) if len(d) else 0
            o = out[out["n_insights"] > 0]; a_, b_ = o.iloc[0], (o.iloc[-2] if len(o) > 1 else o.iloc[-1])
            R["taste_rel"] = dict(n=len(tp), r_words=corr("words"), r_hashtags=corr("hashtags"), likes_with_hashtags=mean(withH), likes_without_hashtags=mean(noH), n_with=len(withH), n_without=len(noH),
                                  y0=int(a_["year"]), y0_hashtags=a_["hashtags"], y0_words=a_["words"], y0_likes=a_["likes_mean"], y1=int(b_["year"]), y1_hashtags=b_["hashtags"], y1_words=b_["words"], y1_likes=b_["likes_mean"],
                                  posts_total=int(out["posts"].sum()), last_year=int(out["year"].max()))
    # concentration of likes on the year's top-5 accounts (self-likes excluded)
    lk = likes[likes["author"].astype(str) != "me"]; prev = None; rows = []
    for y in sorted(lk["year"].unique()):
        l = lk[lk["year"] == y]; cnt = l["author"].value_counts()
        ranked = sorted(cnt.items(), key=lambda e: (-e[1], str(e[0]))); top = ranked[:5]; topn = sum(n for _a, n in top)
        rows.append(dict(year=int(y), n=len(l), accounts=len(cnt), top5_n=topn, share_top5=topn / len(l), returning=sum(a in prev for a, _n in top) if prev is not None else 0, has_prev=prev is not None))
        prev = {a for a, _n in top}
    conc = pd.DataFrame(rows); R["taste_conc"] = conc
    if len(conc):
        f, l_ = conc.iloc[0], conc.iloc[-1]; low = conc.sort_values("share_top5", kind="stable").iloc[0]; t = conc[conc["has_prev"]]
        R["taste_conc_sum"] = dict(first_year=int(f["year"]), first_pct=f["share_top5"], first_accounts=int(f["accounts"]), last_year=int(l_["year"]), last_pct=l_["share_top5"], low_year=int(low["year"]), low_pct=low["share_top5"],
                                   mean_returning=round(t["returning"].mean(), 1) if len(t) else float("nan"), n_transitions=len(t), n_low_return=int((t["returning"] <= 1).sum()),
                                   max_accounts=int(conc["accounts"].max()), max_accounts_year=int(conc.loc[conc["accounts"].idxmax(), "year"]))
    say("Likes still go almost entirely to accounts in the network, but the recommended share is rising: in [taste_rec_sum.first_year] [taste_rec_sum.first_pct] of liked posts came from accounts not followed; over [taste_rec_sum.early_label] as a whole it was [taste_rec_sum.early_pct], over [taste_rec_sum.late_label] [taste_rec_sum.late_pct]. Saves tell the other story: [taste_rec_sum.save_pct] of the [taste_rec_sum.save_n] saved posts are from accounts not followed, and the number of accounts saved from for the first time went from [taste_rec_sum.save_first_new] in [taste_rec_sum.save_first_year] to [taste_rec_sum.save_last_new] in [taste_rec_sum.save_last_year]. The account's own [taste_rel.posts_total] posts stop in [taste_rel.last_year]; between [taste_rel.y0] and [taste_rel.y1] captions shrank from [taste_rel.y0_words] to [taste_rel.y1_words] words and hashtags from [taste_rel.y0_hashtags] to [taste_rel.y1_hashtags] per post while mean likes received rose from [taste_rel.y0_likes] to [taste_rel.y1_likes]; across the [taste_rel.n] posts with insights the correlation of likes with caption words is [taste_rel.r_words] and with hashtags [taste_rel.r_hashtags]: nothing the author wrote predicts the reception. Taste itself de-concentrated and then stopped moving: the five most-liked accounts took [taste_conc_sum.first_pct] of likes in [taste_conc_sum.first_year] and [taste_conc_sum.low_pct] in [taste_conc_sum.low_year], and on average only [taste_conc_sum.mean_returning] of a year's top five were in the previous year's.")
run(calc_taste, needs=["taste"], where="Chapter 12")

In [ ]:
def draw_taste_rec():
    d = R["taste_rec"]; sv = d[d["n_save"] > 0]
    fig, axes = card("Reactions to posts from accounts not followed", "Left: per year, the share of liked and of saved posts whose author is outside the network. Right: accounts saved from for the first time that year, the coloured part outside the network. The network flag is taken at export time, so an account followed later counts as in the network for every year, and these shares understate recommendation in early years. Story reactions are not shown: all of them go to the network.", h=3.2, ncols=2, width_ratios=[58, 42], legend_rows=2, wspace=0.28, pad_top=0.2)
    ax, ax2 = axes[0]
    for nf, f, col in [("n_like", "share_like", C[0]), ("n_save", "share_save", C[1])]:
        p = d[d[nf] > 0]
        ax.plot(p["year"], p[f], color=col, linewidth=2, marker="o", markersize=5)
    ax.set_ylim(0, 1); pct_axis(ax, ticks=[0, .25, .5, .75, 1]); ax.spines["bottom"].set_visible(False); year_ticks(ax, d["year"], every=2 if len(d) > 6 else 1)
    if len(sv):
        ax2.bar(sv["year"], sv["new_saved_accounts"], color=MUTED, width=0.75); ax2.bar(sv["year"], sv["new_saved_rec"], color=C[1], width=0.75)
        year_ticks(ax2, sv["year"], every=2 if len(sv) > 5 else 1); count_axis(ax2, nticks=3); ax2.set_ylim(0, max(1, sv["new_saved_accounts"].max()) * 1.1)
    ax2.spines["bottom"].set_visible(False); ax2.text(0, 1.03, "New accounts saved from", transform=ax2.transAxes, color=INK, fontsize=9)
    legend_below(fig, [("Share of liked posts from accounts not followed", C[0]), ("Share of saved posts from accounts not followed", C[1]), ("New accounts saved from: in the network", MUTED)], ncol=2)
    done(fig, "c-taste-rec")
run(draw_taste_rec, needs=["taste_rec"], where="Chapter 12 chart")

def draw_taste_out():
    o = R["taste_output"]; years = list(o["year"]); pp = R["taste_posts"]
    fig, axes = card("What the account published, and what came back", "Per year of publication: posts, mean caption words and hashtags per post. The last panel puts each post with creator insights by its caption length and the likes it received.", h=2.6, ncols=4, legend_rows=1, wspace=0.35, pad_top=0.2)
    panels = [("posts", "Posts published", lambda v: fmtN(int(v)), C[0]), ("words", "Caption words per post", lambda v: f"{v:.1f}", C[2]), ("hashtags", "Hashtags per post", lambda v: f"{v:.2f}", C[2])]
    for ax, (f, label, fmt, col) in zip(axes[0], panels):
        ax.bar(range(len(years)), o[f], color=col, width=0.75)
        ax.text(0, 1.05, label, transform=ax.transAxes, color=INK, fontsize=8.5)
        ax.set_xticks(range(len(years))); ax.set_xticklabels([str(y)[2:] for y in years]); ax.set_ylim(0, max(1e-9, o[f].max()) * 1.1)
        ax.yaxis.set_major_locator(MaxNLocator(3)); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _p, fmt=fmt: fmt(v))); ax.yaxis.grid(False)
    ax = axes[0][3]
    ax.text(0, 1.05, "Likes received vs caption words", transform=ax.transAxes, color=INK, fontsize=8.5)
    if len(pp):
        ax.scatter(pp["words"], pp["likes"], s=22, c=[C[2] if h > 0 else MUTED for h in pp["hashtags"]], alpha=0.8, linewidths=0)
        ax.set_xlim(0, max(1, pp["words"].max()) * 1.05); ax.set_ylim(0, max(1, pp["likes"].max()) * 1.1)
    ax.yaxis.set_major_locator(MaxNLocator(3)); ax.xaxis.set_major_locator(MaxNLocator(3)); ax.yaxis.grid(False)
    legend_below(fig, [("Post with hashtags", C[2]), ("Post without hashtags", MUTED)])
    done(fig, "c-taste-out")
run(draw_taste_out, needs=["taste_output", "taste_posts"], where="Chapter 12 chart")

def draw_taste_conc():
    d = R["taste_conc"]
    fig, ax = card("How concentrated the liking is, year by year", "Bars: the share of the year's likes (self-likes excluded) that went to its five most-liked accounts. Dots under each year: how many of those five were already in the previous year's top five.", h=3.5, legend_rows=2)
    ax.bar(d["year"], d["share_top5"], color=C[0], width=0.75)
    ax.set_ylim(-0.22, 1); pct_axis(ax, ticks=[0, .25, .5, .75, 1]); ax.spines["bottom"].set_position(("data", 0))
    ax.set_xticks([]); gap = 0.11
    for r in d.itertuples():
        ax.text(r.year, -0.06, str(r.year), ha="center", va="center", color=TICK, fontsize=9)
        if r.has_prev:
            for k in range(5):
                ax.plot([r.year + (k - 2) * gap], [-0.16], "o", markersize=4.5, color=C[1] if k < r.returning else MUTED, alpha=1 if k < r.returning else 0.5)
    legend_below(fig, [("Share of likes to the year's top 5 accounts", C[0]), ("Top-5 account also in previous year's top 5", C[1]), ("New to the top 5", MUTED)], ncol=2)
    done(fig, "c-taste-conc")
run(draw_taste_conc, needs=["taste_conc"], where="Chapter 12 chart")

## Chapter 13 — What the likes were about, and how the comments changed tone

**The economics question.** The recommender does not need to know *who* you are; it needs to know *what* you respond
to. The optional last step of notebook 02 labelled each liked and saved post with a coarse topic from its caption, and
each comment you wrote with a tone — a keyword lexicon run on your own computer, not the platform's classifier. Rough
as they are, these labels show your revealed preferences the way an advertiser sees them: a distribution over
categories that drifts over the years (calc `topics_mix`, `topics_shift`), and the public face you show other users
in your comments (`topics_tone`). The early and late periods compared are set at the top of the cell.

In [ ]:
TOPICS_NONE, TOPICS_REST = "no topical words", "other topics"
TOPICS_EARLY, TOPICS_LATE = (2018, 2020), (2024, 2026)

def calc_topics():
    tp = T["topics"]; has_topic = tp["topic"].notna() & (tp["topic"].astype(str) != "")
    posts = tp[tp["kind"].isin(["like", "save"]) & has_topic]
    top6 = sorted(posts[posts["topic"] != "other"]["topic"].value_counts().items(), key=lambda e: (-e[1], e[0]))[:6]; top6 = [t for t, _n in top6]
    bands = top6 + [TOPICS_REST, TOPICS_NONE]; band = lambda t: TOPICS_NONE if t == "other" else t if t in top6 else TOPICS_REST
    rows = []
    for kind in ["like", "save"]:
        p = posts[posts["kind"] == kind].assign(band=lambda d: d["topic"].map(band))
        for y in sorted(p["year"].unique()):
            py = p[p["year"] == y]; vc = py["band"].value_counts()
            rows += [dict(kind=kind, year=int(y), band=b, n=int(vc.get(b, 0)), total=len(py), share=vc.get(b, 0) / len(py)) for b in bands]
    R["topics_mix"] = pd.DataFrame(rows)
    c = tp[tp["kind"] == "comment"]
    R["topics_tone"] = pd.DataFrame([dict(year=int(y), n=len(cy), positive=int((cy["tone"] == "positive").sum()), neutral=int((cy["tone"] == "neutral").sum()), negative=int((cy["tone"] == "negative").sum()),
                                          mixed=int((cy["tone"] == "mixed").sum()), share_positive=(cy["tone"] == "positive").mean(), words=round(cy["n_words"].sum() / max(1, len(cy)), 1))
                                     for y in sorted(c["year"].unique()) for cy in [c[c["year"] == y]]])
    l = tp[(tp["kind"] == "like") & has_topic]
    early, late = l[(l["year"] >= TOPICS_EARLY[0]) & (l["year"] <= TOPICS_EARLY[1])], l[(l["year"] >= TOPICS_LATE[0]) & (l["year"] <= TOPICS_LATE[1])]
    top7 = [t for t, _n in sorted(l[l["topic"] != "other"]["topic"].value_counts().items(), key=lambda e: (-e[1], e[0]))[:7]]
    band7 = lambda t: TOPICS_NONE if t == "other" else t if t in top7 else TOPICS_REST
    be, bl = early["topic"].map(band7).value_counts(), late["topic"].map(band7).value_counts()
    rows = []
    for b in top7 + [TOPICS_REST, TOPICS_NONE]:
        ne, nl = int(be.get(b, 0)), int(bl.get(b, 0)); se, sl = (ne / len(early) if len(early) else 0), (nl / len(late) if len(late) else 0)
        rows.append(dict(band=b, is_topic=b not in (TOPICS_NONE, TOPICS_REST), n_early=ne, share_early=se, n_late=nl, share_late=sl, diff_pp=round((sl - se) * 100, 1), total_early=len(early), total_late=len(late)))
    shift = pd.DataFrame(rows); R["topics_shift"] = shift
    lt = tp[(tp["kind"] == "like") & has_topic]; none = int((lt["topic"] == "other").sum())
    topE = sorted(lt[lt["topic"] != "other"]["topic"].value_counts().items(), key=lambda e: (-e[1], e[0]))
    t7 = shift[shift["is_topic"]]
    mover = t7.loc[t7["diff_pp"].abs().idxmax()] if len(t7) else None; faller = t7.loc[t7["diff_pp"].idxmin()] if len(t7) else None
    tone = R["topics_tone"]; ty = tone[tone["n"] > 0]; peak = tone.sort_values("n", ascending=False, kind="stable").iloc[0] if len(tone) else None
    fb = tp[tp["kind"] == "feedback"]
    R["topics_sum"] = dict(likes_n=int((tp["kind"] == "like").sum()), likes_captioned=len(lt), none_pct=none / len(lt) if len(lt) else float("nan"),
                           top_topic=topE[0][0] if topE else "—", top_topic_n=topE[0][1] if topE else 0, top_topic_pct=topE[0][1] / len(lt) if topE and len(lt) else float("nan"),
                           early_label=f"{TOPICS_EARLY[0]}–{TOPICS_EARLY[1]}", late_label=f"{TOPICS_LATE[0]}–{TOPICS_LATE[1]}", late_from=TOPICS_LATE[0], early_n=len(early), late_n=len(late),
                           mover=mover["band"] if mover is not None else "—", mover_early_pct=mover["share_early"] if mover is not None else None, mover_late_pct=mover["share_late"] if mover is not None else None,
                           mover_dir=("up" if mover["diff_pp"] >= 0 else "down") if mover is not None else "—", mover_abs_pp=abs(mover["diff_pp"]) if mover is not None else None,
                           faller=faller["band"] if faller is not None else "—", faller_early_pct=faller["share_early"] if faller is not None else None, faller_late_pct=faller["share_late"] if faller is not None else None,
                           comments_n=len(c), positive_pct=(c["tone"] == "positive").mean() if len(c) else float("nan"), neutral_pct=(c["tone"] == "neutral").mean() if len(c) else float("nan"),
                           negative_n=int((c["tone"] == "negative").sum()), negative_pct=(c["tone"] == "negative").mean() if len(c) else float("nan"),
                           comments_peak_year=int(peak["year"]) if peak is not None else None, comments_peak_n=int(peak["n"]) if peak is not None else None,
                           comments_first_year=int(ty.iloc[0]["year"]) if len(ty) else None, comments_last_year=int(ty.iloc[-1]["year"]) if len(ty) else None,
                           comments_last3_n=int(ty[ty["year"] >= TOPICS_LATE[0]]["n"].sum()) if len(ty) else 0,
                           feedback_n=len(fb), feedback_not_interested=int((fb["feedback"] == "Not interested").sum()))
    say("Most liked captions say nothing a topic lexicon can catch: of the [topics_sum.likes_captioned] liked posts with a caption, [topics_sum.none_pct] contain no topical word, and the most frequent topic, [topics_sum.top_topic], covers [topics_sum.top_topic_pct] of them. Between [topics_sum.early_label] ([topics_sum.early_n] captioned likes) and [topics_sum.late_label] ([topics_sum.late_n]) the biggest mover is [topics_sum.mover], [topics_sum.mover_dir] from [topics_sum.mover_early_pct] to [topics_sum.mover_late_pct] of liked posts, while [topics_sum.faller] fell from [topics_sum.faller_early_pct] to [topics_sum.faller_late_pct]; saves, by contrast, cluster on home, work and fashion. The [topics_sum.comments_n] comments the account wrote peak in [topics_sum.comments_peak_year] ([topics_sum.comments_peak_n]) and almost stop afterwards ([topics_sum.comments_last3_n] since [topics_sum.late_from]); [topics_sum.positive_pct] of them read positive, [topics_sum.neutral_pct] neutral and only [topics_sum.negative_n] negative. Explicit taste signals are rarer still: the export holds [topics_sum.feedback_n] feedback events, [topics_sum.feedback_not_interested] of them \"not interested\", too few to chart.")
run(calc_topics, needs=["topics"], where="Chapter 13")

In [ ]:
def draw_topics_mix():
    d = R["topics_mix"]; bands = list(dict.fromkeys(d["band"]))
    col = lambda b: MUTED if b == TOPICS_NONE else C[6] if b == TOPICS_REST else C[bands.index(b)]
    fig, axes = card("What the liked and saved posts were about, year by year", "Share of liked posts (left) and saved posts (right) per year by topic band. A post's topic comes from a keyword lexicon applied to the words and hashtags of its caption: eighteen coarse categories, of which the six most frequent across likes and saves are shown, the other eleven grouped as \"other topics\", and \"no topical words\" for captions that match no list. Posts without a caption (and likes on comments) are left out. The lexicon was run on the student's own computer; no caption text is kept.", h=3.4, ncols=2, width_ratios=[56, 44], legend_rows=2, wspace=0.25, pad_top=0.2)
    for ax, kind, label in zip(axes[0], ["like", "save"], ["Posts liked", "Posts saved"]):
        dk = d[d["kind"] == kind]; years = sorted(dk["year"].unique())
        bottom = pd.Series(0.0, index=years)
        for b in bands:
            s = dk[dk["band"] == b].set_index("year")["share"].reindex(years, fill_value=0)
            ax.bar(years, s.values, bottom=bottom.values, color=col(b), width=0.8); bottom = bottom + s
        ax.set_ylim(0, 1); pct_axis(ax, ticks=[0, .25, .5, .75, 1]); ax.spines["bottom"].set_visible(False)
        ax.text(0, 1.03, label, transform=ax.transAxes, color=INK, fontsize=9)
        if years:
            year_ticks(ax, years, every=2 if len(years) > 5 else 1)
    legend_below(fig, [(b[:1].upper() + b[1:], col(b)) for b in bands], ncol=3)
    done(fig, "c-topics-mix")
run(draw_topics_mix, needs=["topics_mix"], where="Chapter 13 chart")

def draw_topics_tone():
    d = R["topics_tone"]; tones = [("positive", "Positive", C[2]), ("neutral", "Neutral", MUTED), ("mixed", "Mixed", C[4]), ("negative", "Negative", C[3])]
    fig, ax = card("Tone of the comments written, per year", "Comments the account wrote, counted per year and coloured by tone. Tone is assigned by a small word and emoji lexicon (positive, negative, both = mixed, neither = neutral), computed locally on the comment text, which is not kept. Short comments and emoji-only replies tend to land in \"neutral\", so the neutral share is an upper bound.", h=3.3, legend_rows=1)
    bottom = pd.Series(0.0, index=d.index)
    for k, _l, col in tones:
        ax.bar(d["year"], d[k], bottom=bottom, color=col, width=0.75); bottom = bottom + d[k]
    for r in d.itertuples():
        strong(ax, r.year, r.n + max(1, d["n"].max()) * 0.015, fmtN(int(r.n)), ha="center", va="bottom")
    year_ticks(ax, d["year"]); count_axis(ax); ax.spines["bottom"].set_visible(False); ax.set_ylim(0, max(1, d["n"].max()) * 1.15)
    legend_below(fig, [(l, c) for _k, l, c in tones])
    done(fig, "c-topics-tone")
run(draw_topics_tone, needs=["topics_tone"], where="Chapter 13 chart")

def draw_topics_shift():
    d = R["topics_shift"]; y = list(range(len(d)))
    fig, ax = card("Topic shift: liked posts in 2018\u20132020 versus 2024\u20132026", "For the seven topics most frequent among liked posts, plus the two residual bands: the share of captioned liked posts in each period (muted dot: early period; coloured dot: late period, one colour when the share rose and another when it fell, see the legend) and the change in percentage points. The late period holds far fewer likes, so small differences are noise.", h=4, legend_rows=1, pad_top=0.2)
    for i, r in enumerate(d.itertuples()):
        up = r.share_late >= r.share_early; col = C[0] if up else C[1]
        ax.plot([r.share_early, r.share_late], [i, i], color=col, linewidth=2, solid_capstyle="round")
        ax.plot([r.share_early], [i], "o", color=MUTED, markersize=7); ax.plot([r.share_late], [i], "o", color=col, markersize=7)
        strong(ax, max(r.share_early, r.share_late) + 0.008, i, ("+" if r.diff_pp > 0 else "") + f"{r.diff_pp:.0f} pp", va="center")
    ax.set_yticks(y); ax.set_yticklabels([truncate(b) for b in d["band"]]); ax.invert_yaxis()
    for lab, is_topic in zip(ax.get_yticklabels(), d["is_topic"]):
        lab.set_color(TICK if is_topic else MUTED)
    ax.set_xlim(0, max(0.05, d[["share_early", "share_late"]].max().max()) * 1.12); ax.xaxis.set_major_locator(MaxNLocator(4)); pct_axis(ax, "x"); ax.xaxis.tick_top()
    ax.yaxis.grid(False); ax.xaxis.grid(True, color=GRID); ax.spines["bottom"].set_visible(False)
    legend_below(fig, [(f"Share in {TOPICS_EARLY[0]}–{TOPICS_EARLY[1]}", MUTED), (f"Share in {TOPICS_LATE[0]}–{TOPICS_LATE[1]}: up", C[0]), (f"Share in {TOPICS_LATE[0]}–{TOPICS_LATE[1]}: down", C[1])])
    done(fig, "c-topics-shift")
run(draw_topics_shift, needs=["topics_shift"], where="Chapter 13 chart")

## Data-quality notes — what this dataset can and cannot say

Before you write anything from these charts, know the limits of the source. The table below shows, for every table
with a timestamp, how many rows it has and the period it covers; read it with these points in mind.

- **The viewing history is a few days long.** `ads_viewed`, `posts_viewed`, `videos_watched`, `stories_viewed` cover
  roughly the last week before the export (chapters 6, 7 and 10). Likes, saves, messages and follows go back to the
  account's creation. Never compare a count from the first group with one from the second without dividing by the
  period covered.
- **`advertisers.csv` is undated.** It is the state of Meta's matching tables at export time; it does not say when a
  business uploaded your contact, nor whether it ever ran an ad you saw.
- **`in_network` is measured today.** An account counts as "followed" if it is in your lists *now*. A 2019 like on an
  account you followed in 2021 is labelled in-network; the recommended shares of chapters 8 and 12 are therefore lower
  bounds, and the bias grows for older years.
- **Pseudonyms change with every run** of notebook 02 (`user_0417` in your file is not `user_0417` in a classmate's,
  nor in your own next run). Never merge two anonymised exports on pseudonyms. `me` is stable.
- **Captions and messages are metadata only.** Word counts, emoji counts, hashtags, flags; the topic and tone labels of
  chapter 13 come from a keyword lexicon, not from reading the content. Treat them as noisy.
- **Instagram's own numbers are not raw either.** `insights_*` (creator accounts) have gaps and `--` where the platform
  did not count; `likes.csv` mixes liked posts and liked comments (`kind`); timestamps are UTC (converted to Rome time
  here) and the `SHIFT_DAYS` option of notebook 02 may have moved every date by the same unknown offset (hours and
  weekdays survive, calendar dates do not).
- **Absence is not evidence.** A missing table means the export did not contain that file, which happens for features
  never used, for old accounts, and for files Meta has quietly stopped exporting.

In [ ]:
cov = []
for p in sorted(glob.glob(os.path.join(DATA or "", "*.csv"))):
    name = os.path.basename(p)[:-4]
    df = load(name)
    if df is None:
        cov.append((name, 0, None, None, None)); continue
    col = next((c for c in ["timestamp", "first_message"] if c in df.columns), None)
    if col is None:
        cov.append((name, len(df), "undated", "", "")); continue
    t = to_dt(df[col]).dropna()
    if len(t) == 0:
        cov.append((name, len(df), "no valid dates", "", "")); continue
    cov.append((name, len(df), f"{t.min():%Y-%m-%d}", f"{t.max():%Y-%m-%d}", (t.max().normalize() - t.min().normalize()).days + 1))
display(pd.DataFrame(cov, columns=["table", "rows", "first", "last", "days covered"]).set_index("table"))

## Discussion questions

1. **Attention allocation.** Chapter 1 shows how your actions shifted over the years between public (posts, comments,
   likes) and private (messages, story reactions, saves). Who captures the value of each kind of action — you, your
   contacts, the platform, advertisers? Why would a platform *want* the shift you see?
2. **Network effects.** Chapters 3 and 11 show that a few threads carry most of your messages and how they age, and
   chapter 5 how many ties are reciprocal. If your ten busiest threads moved to another app tomorrow, what would keep
   you on Instagram? Is the lock-in in the graph, in the content, or in the habit (chapter 4)?
3. **Recommender vs social graph.** Chapters 6, 8 and 12 estimate the share of your engagement that came from accounts
   you do not follow, and what the feed showed you last week. Who decides what you see — you, your network, or the
   ranking model? What changed in the platform's incentives when the recommender took over from the follow graph, and
   who gained and lost (users, small creators, large creators, advertisers)?
4. **Targeted advertising and customer lists.** Chapter 7 lists businesses that uploaded your contact details to Meta.
   You gave your e-mail to a shop, not to Instagram: through which transaction did it end up here, who was paid, and is
   there a market for this match? Would the ads you saw be worth less without it?
5. **Creator economy and disclosure.** Chapters 9, 10 and 13 show the other side of the counter, the ad supply chain
   and the topics you respond to. If a creator you follow is paid to post, who bears the cost of *not* disclosing — the
   follower, the honest competitor, the platform? Does the `#adv` hashtag fix the information problem?

## What to hand in

Pick **three charts** from this notebook (yours, or re-drawn with a change you justify) and write a **one-page memo**
(max 500 words): for each chart, one paragraph on what it shows in your data and one on what it says about the
economics of the platform, citing at least one concept from the course. Paste the charts in the memo; do not attach the
data. Everything in the charts is aggregated and pseudonymised — if a chart would let a reader identify a person,
choose another one.